# Lab 03 — Quantization: predict bytes, inspect error, compare paired outputs

Start with the literal weights and input vector, not a compression slogan. Then inspect the **recorded** paired prompts and token IDs. Finally run your own bounded FP16/NF4 smoke on the same pinned CUDA environment. CPU replays do not replace required GPU evidence.

Read [Book 03](https://levshaazz.github.io/llm-serving-mastery/en/book/03/) and the [student guide](https://levshaazz.github.io/llm-serving-mastery/en/start/). Students independently arrange a CUDA GPU. Ask private diagnostic questions in your repository's GitHub Issue mentioning @levshaazz; no secrets in public issues.

**Unchanged workload:** pinned Qwen2.5-0.5B, two separate format loads, four literal prompts, one natural-EOS sample and three forced-24-token timing trials per prompt, plus one excluded warmup per format. Protocol v3 adds exact IDs, checkpoints and validation; it does not change this workload. It is not a quality benchmark, HTTP measurement or judge run.


## 0. Complete quantizer — CPU only, no install or network

The invented toy uses W[out,in]=[2,4], x[in]=[4], scale 0.25, zero-point 0 and signed INT4 [-8,7]. Compute round-to-nearest **ties-to-even**, clip, reconstruct and the signed error reconstruction−original. The zero-point implementation also supports a separate affine example.

Predict first: does 2.25 become 2.25? Why is 1.83's error different? Does each output error equal the sum of input-weighted weight errors? Packing puts element 2i in the **low nibble**, two's-complement signed encoding, with explicit logical length. This pedagogical layout is **not** bitsandbytes NF4 packing.


In [ ]:
toy_namespace={'__name__':'topic03_cpu_toy'}
exec("\"\"\"Complete, deliberately invented INT4 arithmetic. Standard library; never NF4 kernels.\"\"\"\nimport math\n\n# bitsandbytes 0.50.2 functional.get_4bit_type('nf4'); lookup values, not byte layout.\nNF4 = [-1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453,\n       -0.28444138169288635, -0.18477343022823334, -0.09105003625154495, 0.0,\n       0.07958029955625534, 0.16093020141124725, 0.24611230194568634,\n       0.33791524171829224, 0.44070982933044434, 0.5626170039176941,\n       0.7229568362236023, 1.0]\n\n\ndef scalar(value, scale, zero_point=0):\n    if not math.isfinite(value) or not math.isfinite(scale) or scale <= 0:\n        raise ValueError('finite value and positive finite scale required')\n    if type(zero_point) is not int or not -8 <= zero_point <= 7:\n        raise ValueError('signed INT4 zero point required')\n    scaled = value / scale\n    rounded = round(scaled)  # Python: nearest, ties to even, including negatives.\n    shifted = rounded + zero_point\n    q = max(-8, min(7, shifted))\n    reconstructed = scale * (q - zero_point)\n    return dict(value=value, scale=scale, zero_point=zero_point, scaled=scaled,\n                rounded=rounded, shifted=shifted, clipped=shifted != q, q=q,\n                reconstructed=reconstructed, error=reconstructed-value)\n\n\ndef pack_int4(codes):\n    \"\"\"Toy two's-complement encoding: element 2i in LOW nibble. Not bnb layout.\"\"\"\n    if any(type(q) is not int or not -8 <= q <= 7 for q in codes):\n        raise ValueError('codes must be signed INT4 integers')\n    return [(codes[i] & 15) | (((codes[i+1] if i+1 < len(codes) else 0) & 15) << 4)\n            for i in range(0, len(codes), 2)]\n\n\ndef unpack_int4(packed, count):\n    if type(count) is not int or count < 0 or len(packed) != (count+1)//2:\n        raise ValueError('logical element count must match packed length')\n    if any(type(b) is not int or not 0 <= b <= 255 for b in packed):\n        raise ValueError('bytes must be integers in [0,255]')\n    values = [n-16 if n >= 8 else n for b in packed for n in (b & 15, b >> 4)]\n    if count % 2 and values[-1] != 0:\n        raise ValueError('toy padding nibble must be zero')\n    return values[:count]\n\n\ndef matvec(weights, inputs):\n    if not weights or any(len(row) != len(inputs) for row in weights):\n        raise ValueError('W[out,in] and x[in] must align')\n    return [sum(w*x for w, x in zip(row, inputs)) for row in weights]\n\n\ndef group_quantize(values, group_size):\n    if type(group_size) is not int or group_size <= 0:\n        raise ValueError('positive group size required')\n    if not values or any(not math.isfinite(v) for v in values):\n        raise ValueError('nonempty finite values required')\n    groups = []\n    for offset in range(0, len(values), group_size):\n        group = values[offset:offset+group_size]\n        # Symmetric absmax/7 intentionally leaves -8 unused: state the convention.\n        scale = max(abs(v) for v in group)/7 or 1.0\n        groups.append(dict(offset=offset, scale=scale, rows=[scalar(v, scale) for v in group]))\n    reconstructed = [r['reconstructed'] for g in groups for r in g['rows']]\n    return dict(groups=groups, reconstructed=reconstructed,\n                mse=sum((a-b)**2 for a,b in zip(values,reconstructed))/len(values),\n                payload_bytes=(len(values)+1)//2, fp16_scale_bytes=2*len(groups))\n\n\ndef toy_example():\n    weights = [[-1.83, -.52, .24, 2.25], [.13, -.88, 1.12, -2.40]]\n    inputs = [1.0, -2.0, .5, 1.5]\n    rows = [[scalar(value, .25) for value in row] for row in weights]\n    reconstructed = [[r['reconstructed'] for r in row] for row in rows]\n    reference, output = matvec(weights, inputs), matvec(reconstructed, inputs)\n    codes = [r['q'] for row in rows for r in row]\n    packed = pack_int4(codes)\n    outliers = [-.1, .2, -.3, .4, 8.0, 9.0, 10.0, 11.0]\n    nf4_inputs = [-.5, -.1, .0, .1, .25, .5]\n    return dict(schema='topic03-int4-toy-v1', kind='illustrative; invented weights, not Qwen or NF4',\n                rounding='nearest, ties to even', error_convention='reconstructed minus original',\n                scalar_examples=[scalar(1.83,.25), scalar(2.25,.25)],\n                ties=[dict(scaled=x,rounded=round(x)) for x in [-2.5,-1.5,1.5,2.5]],\n                affine_example=scalar(.6,.2,-2), weights=weights, input=inputs, scale=.25,\n                rows=rows, codes=codes, packed_bytes=packed,\n                packed_hex=[f'{b:02x}' for b in packed], unpacked=unpack_int4(packed,len(codes)),\n                packing='two complement, low nibble first; logical count stored separately; NOT bnb packing',\n                reconstructed=reconstructed, reference_output=reference, quantized_output=output,\n                output_error=[b-a for a,b in zip(reference,output)],\n                output_error_terms=[[(r['error']*x) for r,x in zip(row,inputs)] for row in rows],\n                second_input=[0.,0.,0.,1.], second_output_error=[row[-1]['error'] for row in rows],\n                outliers=dict(values=outliers,global_scale=group_quantize(outliers,8),\n                              group4=group_quantize(outliers,4), scale_policy='absmax/7; zero group uses1'),\n                nf4=dict(source='https://github.com/bitsandbytes-foundation/bitsandbytes/blob/0.50.2/bitsandbytes/functional.py',\n                         codebook=NF4, description='normalized nearest-code lookup only; not bnb packing or CUDA kernel',\n                         scaled_example=dict(weight=.5,absmax=2.,normalized=.25,index=10,\n                                             reconstructed=2*NF4[10],error=2*NF4[10]-.5),\n                         examples=[dict(value=v,index=min(range(16),key=lambda i:abs(NF4[i]-v)),\n                                        reconstructed=min(NF4,key=lambda c:abs(c-v))) for v in nf4_inputs]))\n",toy_namespace)
TOY=toy_namespace['toy_example']()
print(TOY['kind'])
print('W[out,in]:',TOY['weights'],'x[in]:',TOY['input'])
for output_index,row in enumerate(TOY['rows']):
    for input_index,record in enumerate(row): print(output_index,input_index,record)
print('Integer codes:',TOY['codes'],'packed hex:',TOY['packed_hex'])
assert TOY['unpacked']==TOY['codes']
print('Reference output:',TOY['reference_output'])
print('Reconstructed output:',TOY['quantized_output'],'signed errors:',TOY['output_error'])
for terms,error in zip(TOY['output_error_terms'],TOY['output_error']):
    assert abs(sum(terms)-error)<1e-12
    print('Input-weighted error terms:',terms,'sum:',sum(terms))
print('Ties:',TOY['ties'],'separate affine example:',TOY['affine_example'])


### Outlier groups and NF4 are different questions

Keep the same eight weights and compare one global absmax/7 scale with two groups of four. Count four payload bytes plus two versus four FP16 scale bytes. On a tiny tensor the metadata can erase the payload advantage; the example intentionally exposes it. This scale policy leaves signed code −8 unused and uses scale 1 for an all-zero group.

NF4 chooses from a nonuniform 16-entry normalized codebook. The next cell is a CPU nearest-code explanation, **not** bitsandbytes block quantization, byte layout, CUDA execution or evidence of speed. The real experiment below explicitly uses NF4 W4A16 with FP16 compute and double quantization off.


In [ ]:
for name in ('global_scale','group4'):
    record=TOY['outliers'][name]
    print(name,'scales:',[g['scale'] for g in record['groups']])
    print('reconstructed:',record['reconstructed'],'MSE:',record['mse'])
    print('payload bytes:',record['payload_bytes'],'FP16 scale bytes:',record['fp16_scale_bytes'])
print('NF4 source:',TOY['nf4']['source'])
print('Codebook:',TOY['nf4']['codebook'])
for row in TOY['nf4']['examples']: print(row)
print('Separate invented scaled NF4 lookup:',TOY['nf4']['scaled_example'])
print('Change x to',TOY['second_input'],'and output error changes to',TOY['second_output_error'])


## 1. Recorded real pair — reveal the input, then compare outputs

The following is embedded instructor evidence, no inference. Choose CASE 0–3. Compare the full rendered chat and all input IDs across formats before comparing output. Natural-EOS quality samples can stop early; timing trials force 24 IDs and may violate one-word/integer instructions. **Never score forced continuations as instruction following.**

Classify each natural sample as valid, incorrect, truncated or needs human review, with your reason. Four tasks cannot estimate production accuracy. First generated-ID callback is not first visible character; EOS counts as a selected ID without visible text.


In [ ]:
import json
RECORDED_QUANTIZATION=json.loads("{\n  \"contract\": \"topic-03-paired-smoke-v3\",\n  \"status\": \"complete\",\n  \"utc_started\": \"2026-10-01T22:01:52.341335+00:00\",\n  \"model_id\": \"Qwen/Qwen2.5-0.5B-Instruct\",\n  \"revision\": \"7ae557604adf67be50417f59c2c2f167def9a775\",\n  \"gpu\": \"NVIDIA GeForce RTX 5070 Ti\",\n  \"torch\": \"2.14.0+cu130\",\n  \"cuda\": \"13.0\",\n  \"transformers\": \"4.51.3\",\n  \"bitsandbytes\": \"0.50.2\",\n  \"trials\": 3,\n  \"max_new_tokens\": 24,\n  \"quality_mode\": \"natural_eos\",\n  \"timing_mode\": \"fixed_output_length\",\n  \"formats\": {\n    \"fp16\": {\n      \"load_s\": 6.734631741001067,\n      \"rows\": [\n        {\n          \"bucket\": \"format\",\n          \"prompt\": \"Reply with exactly one word: blue or orange. What color is the sky on a clear day?\",\n          \"quality_sample\": {\n            \"prompt_tokens\": 49,\n            \"input_token_ids\": [\n              151644,\n              8948,\n              198,\n              2610,\n              525,\n              1207,\n              16948,\n              11,\n              3465,\n              553,\n              54364,\n              14817,\n              13,\n              1446,\n              525,\n              264,\n              10950,\n              17847,\n              13,\n              151645,\n              198,\n              151644,\n              872,\n              198,\n              20841,\n              448,\n              6896,\n              825,\n              3409,\n              25,\n              6303,\n              476,\n              18575,\n              13,\n              3555,\n              1894,\n              374,\n              279,\n              12884,\n              389,\n              264,\n              2797,\n              1899,\n              30,\n              151645,\n              198,\n              151644,\n              77091,\n              198\n            ],\n            \"rendered_chat\": \"<|im_start|>system\\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\\n<|im_start|>user\\nReply with exactly one word: blue or orange. What color is the sky on a clear day?<|im_end|>\\n<|im_start|>assistant\\n\",\n            \"output_tokens\": 2,\n            \"output_token_ids\": [\n              12203,\n              151645\n            ],\n            \"output\": \"blue\",\n            \"output_token_pieces\": [\n              \"blue\",\n              \"<|im_end|>\"\n            ],\n            \"stop_reason\": \"eos\",\n            \"ttft_s\": 0.09361864400125341,\n            \"wall_s\": 0.11512070600110746,\n            \"post_first_s_per_token\": 0.02150206199985405\n          },\n          \"timing_trials\": [\n            {\n              \"prompt_tokens\": 49,\n              \"input_token_ids\": [\n                151644,\n                8948,\n                198,\n                2610,\n                525,\n                1207,\n                16948,\n                11,\n                3465,\n                553,\n                54364,\n                14817,\n                13,\n                1446,\n                525,\n                264,\n                10950,\n                17847,\n                13,\n                151645,\n                198,\n                151644,\n                872,\n                198,\n                20841,\n                448,\n                6896,\n                825,\n                3409,\n                25,\n                6303,\n                476,\n                18575,\n                13,\n                3555,\n                1894,\n                374,\n                279,\n                12884,\n                389,\n                264,\n                2797,\n                1899,\n                30,\n                151645,\n                198,\n                151644,\n                77091,\n                198\n              ],\n              \"rendered_chat\": \"<|im_start|>system\\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\\n<|im_start|>user\\nReply with exactly one word: blue or orange. What color is the sky on a clear day?<|im_end|>\\n<|im_start|>assistant\\n\",\n              \"output_tokens\": 24,\n              \"output_token_ids\": [\n                12203,\n                13,\n                576,\n                12884,\n                7952,\n                6303,\n                1576,\n                432,\n                25963,\n                39020,\n                323,\n                90011,\n                1429,\n                315,\n                279,\n                1008,\n                7987,\n                3042,\n                304,\n                279,\n                16566,\n                13,\n                1096,\n                24844\n              ],\n              \"output\": \"blue. The sky appears blue because it reflects sunlight and absorbs most of the other colors present in the atmosphere. This phenomenon\",\n              \"output_token_pieces\": [\n                \"blue\",\n                \".\",\n                \"ĠThe\",\n                \"Ġsky\",\n                \"Ġappears\",\n                \"Ġblue\",\n                \"Ġbecause\",\n                \"Ġit\",\n                \"Ġreflects\",\n                \"Ġsunlight\",\n                \"Ġand\",\n                \"Ġabsorbs\",\n                \"Ġmost\",\n                \"Ġof\",\n                \"Ġthe\",\n                \"Ġother\",\n                \"Ġcolors\",\n                \"Ġpresent\",\n                \"Ġin\",\n                \"Ġthe\",\n                \"Ġatmosphere\",\n                \".\",\n                \"ĠThis\",\n                \"Ġphenomenon\"\n              ],\n              \"stop_reason\": \"fixed_length\",\n              \"ttft_s\": 0.09547275100158004,\n              \"wall_s\": 0.6731598770002165,\n              \"post_first_s_per_token\": 0.025116831565158107\n            },\n            {\n              \"prompt_tokens\": 49,\n              \"input_token_ids\": [\n                151644,\n                8948,\n                198,\n                2610,\n                525,\n                1207,\n                16948,\n                11,\n                3465,\n                553,\n                54364,\n                14817,\n                13,\n                1446,\n                525,\n                264,\n                10950,\n                17847,\n                13,\n                151645,\n                198,\n                151644,\n                872,\n                198,\n                20841,\n                448,\n                6896,\n                825,\n                3409,\n                25,\n                6303,\n                476,\n                18575,\n                13,\n                3555,\n                1894,\n                374,\n                279,\n                12884,\n                389,\n                264,\n                2797,\n                1899,\n                30,\n                151645,\n                198,\n                151644,\n                77091,\n                198\n              ],\n              \"rendered_chat\": \"<|im_start|>system\\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\\n<|im_start|>user\\nReply with exactly one word: blue or orange. What color is the sky on a clear day?<|im_end|>\\n<|im_start|>assistant\\n\",\n              \"output_tokens\": 24,\n              \"output_token_ids\": [\n                12203,\n                13,\n                576,\n                12884,\n                7952,\n                6303,\n                1576,\n                432,\n                25963,\n                39020,\n                323,\n                90011,\n                1429,\n                315,\n                279,\n                1008,\n                7987,\n                3042,\n                304,\n                279,\n                16566,\n                13,\n                1096,\n                24844\n              ],\n              \"output\": \"blue. The sky appears blue because it reflects sunlight and absorbs most of the other colors present in the atmosphere. This phenomenon\",\n              \"output_token_pieces\": [\n                \"blue\",\n                \".\",\n                \"ĠThe\",\n                \"Ġsky\",\n                \"Ġappears\",\n                \"Ġblue\",\n                \"Ġbecause\",\n                \"Ġit\",\n                \"Ġreflects\",\n                \"Ġsunlight\",\n                \"Ġand\",\n                \"Ġabsorbs\",\n                \"Ġmost\",\n                \"Ġof\",\n                \"Ġthe\",\n                \"Ġother\",\n                \"Ġcolors\",\n                \"Ġpresent\",\n                \"Ġin\",\n                \"Ġthe\",\n                \"Ġatmosphere\",\n                \".\",\n                \"ĠThis\",\n                \"Ġphenomenon\"\n              ],\n              \"stop_reason\": \"fixed_length\",\n              \"ttft_s\": 0.09520329099905211,\n              \"wall_s\": 0.6243529460007267,\n              \"post_first_s_per_token\": 0.02300650673920324\n            },\n            {\n              \"prompt_tokens\": 49,\n              \"input_token_ids\": [\n                151644,\n                8948,\n                198,\n                2610,\n                525,\n                1207,\n                16948,\n                11,\n                3465,\n                553,\n                54364,\n                14817,\n                13,\n                1446,\n                525,\n                264,\n                10950,\n                17847,\n                13,\n                151645,\n                198,\n                151644,\n                872,\n                198,\n                20841,\n                448,\n                6896,\n                825,\n                3409,\n                25,\n                6303,\n                476,\n                18575,\n                13,\n                3555,\n                1894,\n                374,\n                279,\n                12884,\n                389,\n                264,\n                2797,\n                1899,\n                30,\n                151645,\n                198,\n                151644,\n                77091,\n                198\n              ],\n              \"rendered_chat\": \"<|im_start|>system\\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\\n<|im_start|>user\\nReply with exactly one word: blue or orange. What color is the sky on a clear day?<|im_end|>\\n<|im_start|>assistant\\n\",\n              \"output_tokens\": 24,\n              \"output_token_ids\": [\n                12203,\n                13,\n                576,\n                12884,\n                7952,\n                6303,\n                1576,\n                432,\n                25963,\n                39020,\n                323,\n                90011,\n                1429,\n                315,\n                279,\n                1008,\n                7987,\n                3042,\n                304,\n                279,\n                16566,\n                13,\n                1096,\n                24844\n              ],\n              \"output\": \"blue. The sky appears blue because it reflects sunlight and absorbs most of the other colors present in the atmosphere. This phenomenon\",\n              \"output_token_pieces\": [\n                \"blue\",\n                \".\",\n                \"ĠThe\",\n                \"Ġsky\",\n                \"Ġappears\",\n                \"Ġblue\",\n                \"Ġbecause\",\n                \"Ġit\",\n                \"Ġreflects\",\n                \"Ġsunlight\",\n                \"Ġand\",\n                \"Ġabsorbs\",\n                \"Ġmost\",\n                \"Ġof\",\n                \"Ġthe\",\n                \"Ġother\",\n                \"Ġcolors\",\n                \"Ġpresent\",\n                \"Ġin\",\n                \"Ġthe\",\n                \"Ġatmosphere\",\n                \".\",\n                \"ĠThis\",\n                \"Ġphenomenon\"\n              ],\n              \"stop_reason\": \"fixed_length\",\n              \"ttft_s\": 0.07935376000023098,\n              \"wall_s\": 0.6523988490007468,\n              \"post_first_s_per_token\": 0.024915003869587646\n            }\n          ],\n          \"p95_ttft_s\": 0.09547275100158004,\n          \"median_ttft_s\": 0.09520329099905211,\n          \"p95_wall_s\": 0.6731598770002165,\n          \"median_wall_s\": 0.6523988490007468\n        },\n        {\n          \"bucket\": \"arithmetic\",\n          \"prompt\": \"Compute 17 + 28. Give only the integer.\",\n          \"quality_sample\": {\n            \"prompt_tokens\": 43,\n            \"input_token_ids\": [\n              151644,\n              8948,\n              198,\n              2610,\n              525,\n              1207,\n              16948,\n              11,\n              3465,\n              553,\n              54364,\n              14817,\n              13,\n              1446,\n              525,\n              264,\n              10950,\n              17847,\n              13,\n              151645,\n              198,\n              151644,\n              872,\n              198,\n              46254,\n              220,\n              16,\n              22,\n              488,\n              220,\n              17,\n              23,\n              13,\n              20678,\n              1172,\n              279,\n              7546,\n              13,\n              151645,\n              198,\n              151644,\n              77091,\n              198\n            ],\n            \"rendered_chat\": \"<|im_start|>system\\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\\n<|im_start|>user\\nCompute 17 + 28. Give only the integer.<|im_end|>\\n<|im_start|>assistant\\n\",\n            \"output_tokens\": 3,\n            \"output_token_ids\": [\n              19,\n              20,\n              151645\n            ],\n            \"output\": \"45\",\n            \"output_token_pieces\": [\n              \"4\",\n              \"5\",\n              \"<|im_end|>\"\n            ],\n            \"stop_reason\": \"eos\",\n            \"ttft_s\": 0.09496427999874868,\n            \"wall_s\": 0.18568834100005915,\n            \"post_first_s_per_token\": 0.04536203050065524\n          },\n          \"timing_trials\": [\n            {\n              \"prompt_tokens\": 43,\n              \"input_token_ids\": [\n                151644,\n                8948,\n                198,\n                2610,\n                525,\n                1207,\n                16948,\n                11,\n                3465,\n                553,\n                54364,\n                14817,\n                13,\n                1446,\n                525,\n                264,\n                10950,\n                17847,\n                13,\n                151645,\n                198,\n                151644,\n                872,\n                198,\n                46254,\n                220,\n                16,\n                22,\n                488,\n                220,\n                17,\n                23,\n                13,\n                20678,\n                1172,\n                279,\n                7546,\n                13,\n                151645,\n                198,\n                151644,\n                77091,\n                198\n              ],\n              \"rendered_chat\": \"<|im_start|>system\\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\\n<|im_start|>user\\nCompute 17 + 28. Give only the integer.<|im_end|>\\n<|im_start|>assistant\\n\",\n              \"output_tokens\": 24,\n              \"output_token_ids\": [\n                19,\n                20,\n                13,\n                15,\n                271,\n                785,\n                2629,\n                315,\n                220,\n                16,\n                22,\n                323,\n                220,\n                17,\n                23,\n                374,\n                220,\n                19,\n                20,\n                13,\n                15,\n                13,\n                8704,\n                582\n              ],\n              \"output\": \"45.0\\n\\nThe sum of 17 and 28 is 45.0. Since we\",\n              \"output_token_pieces\": [\n                \"4\",\n                \"5\",\n                \".\",\n                \"0\",\n                \"ĊĊ\",\n                \"The\",\n                \"Ġsum\",\n                \"Ġof\",\n                \"Ġ\",\n                \"1\",\n                \"7\",\n                \"Ġand\",\n                \"Ġ\",\n                \"2\",\n                \"8\",\n                \"Ġis\",\n                \"Ġ\",\n                \"4\",\n                \"5\",\n                \".\",\n                \"0\",\n                \".\",\n                \"ĠSince\",\n                \"Ġwe\"\n              ],\n              \"stop_reason\": \"fixed_length\",\n              \"ttft_s\": 0.07326022499910323,\n              \"wall_s\": 0.6951976389991614,\n              \"post_first_s_per_token\": 0.027040757130437312\n            },\n            {\n              \"prompt_tokens\": 43,\n              \"input_token_ids\": [\n                151644,\n                8948,\n                198,\n                2610,\n                525,\n                1207,\n                16948,\n                11,\n                3465,\n                553,\n                54364,\n                14817,\n                13,\n                1446,\n                525,\n                264,\n                10950,\n                17847,\n                13,\n                151645,\n                198,\n                151644,\n                872,\n                198,\n                46254,\n                220,\n                16,\n                22,\n                488,\n                220,\n                17,\n                23,\n                13,\n                20678,\n                1172,\n                279,\n                7546,\n                13,\n                151645,\n                198,\n                151644,\n                77091,\n                198\n              ],\n              \"rendered_chat\": \"<|im_start|>system\\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\\n<|im_start|>user\\nCompute 17 + 28. Give only the integer.<|im_end|>\\n<|im_start|>assistant\\n\",\n              \"output_tokens\": 24,\n              \"output_token_ids\": [\n                19,\n                20,\n                13,\n                15,\n                271,\n                785,\n                2629,\n                315,\n                220,\n                16,\n                22,\n                323,\n                220,\n                17,\n                23,\n                374,\n                220,\n                19,\n                20,\n                13,\n                15,\n                13,\n                8704,\n                582\n              ],\n              \"output\": \"45.0\\n\\nThe sum of 17 and 28 is 45.0. Since we\",\n              \"output_token_pieces\": [\n                \"4\",\n                \"5\",\n                \".\",\n                \"0\",\n                \"ĊĊ\",\n                \"The\",\n                \"Ġsum\",\n                \"Ġof\",\n                \"Ġ\",\n                \"1\",\n                \"7\",\n                \"Ġand\",\n                \"Ġ\",\n                \"2\",\n                \"8\",\n                \"Ġis\",\n                \"Ġ\",\n                \"4\",\n                \"5\",\n                \".\",\n                \"0\",\n                \".\",\n                \"ĠSince\",\n                \"Ġwe\"\n              ],\n              \"stop_reason\": \"fixed_length\",\n              \"ttft_s\": 0.04731913999967219,\n              \"wall_s\": 0.8219553260005341,\n              \"post_first_s_per_token\": 0.03367983417395052\n            },\n            {\n              \"prompt_tokens\": 43,\n              \"input_token_ids\": [\n                151644,\n                8948,\n                198,\n                2610,\n                525,\n                1207,\n                16948,\n                11,\n                3465,\n                553,\n                54364,\n                14817,\n                13,\n                1446,\n                525,\n                264,\n                10950,\n                17847,\n                13,\n                151645,\n                198,\n                151644,\n                872,\n                198,\n                46254,\n                220,\n                16,\n                22,\n                488,\n                220,\n                17,\n                23,\n                13,\n                20678,\n                1172,\n                279,\n                7546,\n                13,\n                151645,\n                198,\n                151644,\n                77091,\n                198\n              ],\n              \"rendered_chat\": \"<|im_start|>system\\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\\n<|im_start|>user\\nCompute 17 + 28. Give only the integer.<|im_end|>\\n<|im_start|>assistant\\n\",\n              \"output_tokens\": 24,\n              \"output_token_ids\": [\n                19,\n                20,\n                13,\n                15,\n                271,\n                785,\n                2629,\n                315,\n                220,\n                16,\n                22,\n                323,\n                220,\n                17,\n                23,\n                374,\n                220,\n                19,\n                20,\n                13,\n                15,\n                13,\n                8704,\n                582\n              ],\n              \"output\": \"45.0\\n\\nThe sum of 17 and 28 is 45.0. Since we\",\n              \"output_token_pieces\": [\n                \"4\",\n                \"5\",\n                \".\",\n                \"0\",\n                \"ĊĊ\",\n                \"The\",\n                \"Ġsum\",\n                \"Ġof\",\n                \"Ġ\",\n                \"1\",\n                \"7\",\n                \"Ġand\",\n                \"Ġ\",\n                \"2\",\n                \"8\",\n                \"Ġis\",\n                \"Ġ\",\n                \"4\",\n                \"5\",\n                \".\",\n                \"0\",\n                \".\",\n                \"ĠSince\",\n                \"Ġwe\"\n              ],\n              \"stop_reason\": \"fixed_length\",\n              \"ttft_s\": 0.04982895200009807,\n              \"wall_s\": 0.7673714209995524,\n              \"post_first_s_per_token\": 0.031197498652150185\n            }\n          ],\n          \"p95_ttft_s\": 0.07326022499910323,\n          \"median_ttft_s\": 0.04982895200009807,\n          \"p95_wall_s\": 0.8219553260005341,\n          \"median_wall_s\": 0.7673714209995524\n        },\n        {\n          \"bucket\": \"instruction\",\n          \"prompt\": \"Write exactly three comma-separated verbs for measuring a service.\",\n          \"quality_sample\": {\n            \"prompt_tokens\": 40,\n            \"input_token_ids\": [\n              151644,\n              8948,\n              198,\n              2610,\n              525,\n              1207,\n              16948,\n              11,\n              3465,\n              553,\n              54364,\n              14817,\n              13,\n              1446,\n              525,\n              264,\n              10950,\n              17847,\n              13,\n              151645,\n              198,\n              151644,\n              872,\n              198,\n              7985,\n              6896,\n              2326,\n              31683,\n              72692,\n              61846,\n              369,\n              28990,\n              264,\n              2473,\n              13,\n              151645,\n              198,\n              151644,\n              77091,\n              198\n            ],\n            \"rendered_chat\": \"<|im_start|>system\\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\\n<|im_start|>user\\nWrite exactly three comma-separated verbs for measuring a service.<|im_end|>\\n<|im_start|>assistant\\n\",\n            \"output_tokens\": 24,\n            \"output_token_ids\": [\n              16,\n              13,\n              34104,\n              279,\n              882,\n              4429,\n              311,\n              4583,\n              279,\n              3383,\n              624,\n              17,\n              13,\n              4248,\n              421,\n              279,\n              2473,\n              374,\n              30201,\n              438,\n              3601,\n              624,\n              18,\n              13\n            ],\n            \"output\": \"1. Measure the time taken to complete the task.\\n2. Check if the service is functioning as expected.\\n3.\",\n            \"output_token_pieces\": [\n              \"1\",\n              \".\",\n              \"ĠMeasure\",\n              \"Ġthe\",\n              \"Ġtime\",\n              \"Ġtaken\",\n              \"Ġto\",\n              \"Ġcomplete\",\n              \"Ġthe\",\n              \"Ġtask\",\n              \".Ċ\",\n              \"2\",\n              \".\",\n              \"ĠCheck\",\n              \"Ġif\",\n              \"Ġthe\",\n              \"Ġservice\",\n              \"Ġis\",\n              \"Ġfunctioning\",\n              \"Ġas\",\n              \"Ġexpected\",\n              \".Ċ\",\n              \"3\",\n              \".\"\n            ],\n            \"stop_reason\": \"length\",\n            \"ttft_s\": 0.03439114799948584,\n            \"wall_s\": 0.7953491419993952,\n            \"post_first_s_per_token\": 0.03308513017390911\n          },\n          \"timing_trials\": [\n            {\n              \"prompt_tokens\": 40,\n              \"input_token_ids\": [\n                151644,\n                8948,\n                198,\n                2610,\n                525,\n                1207,\n                16948,\n                11,\n                3465,\n                553,\n                54364,\n                14817,\n                13,\n                1446,\n                525,\n                264,\n                10950,\n                17847,\n                13,\n                151645,\n                198,\n                151644,\n                872,\n                198,\n                7985,\n                6896,\n                2326,\n                31683,\n                72692,\n                61846,\n                369,\n                28990,\n                264,\n                2473,\n                13,\n                151645,\n                198,\n                151644,\n                77091,\n                198\n              ],\n              \"rendered_chat\": \"<|im_start|>system\\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\\n<|im_start|>user\\nWrite exactly three comma-separated verbs for measuring a service.<|im_end|>\\n<|im_start|>assistant\\n\",\n              \"output_tokens\": 24,\n              \"output_token_ids\": [\n                16,\n                13,\n                34104,\n                279,\n                882,\n                4429,\n                311,\n                4583,\n                279,\n                3383,\n                624,\n                17,\n                13,\n                4248,\n                421,\n                279,\n                2473,\n                374,\n                30201,\n                438,\n                3601,\n                624,\n                18,\n                13\n              ],\n              \"output\": \"1. Measure the time taken to complete the task.\\n2. Check if the service is functioning as expected.\\n3.\",\n              \"output_token_pieces\": [\n                \"1\",\n                \".\",\n                \"ĠMeasure\",\n                \"Ġthe\",\n                \"Ġtime\",\n                \"Ġtaken\",\n                \"Ġto\",\n                \"Ġcomplete\",\n                \"Ġthe\",\n                \"Ġtask\",\n                \".Ċ\",\n                \"2\",\n                \".\",\n                \"ĠCheck\",\n                \"Ġif\",\n                \"Ġthe\",\n                \"Ġservice\",\n                \"Ġis\",\n                \"Ġfunctioning\",\n                \"Ġas\",\n                \"Ġexpected\",\n                \".Ċ\",\n                \"3\",\n                \".\"\n              ],\n              \"stop_reason\": \"fixed_length\",\n              \"ttft_s\": 0.05025108600057138,\n              \"wall_s\": 0.6263703559998248,\n              \"post_first_s_per_token\": 0.02504866391301102\n            },\n            {\n              \"prompt_tokens\": 40,\n              \"input_token_ids\": [\n                151644,\n                8948,\n                198,\n                2610,\n                525,\n                1207,\n                16948,\n                11,\n                3465,\n                553,\n                54364,\n                14817,\n                13,\n                1446,\n                525,\n                264,\n                10950,\n                17847,\n                13,\n                151645,\n                198,\n                151644,\n                872,\n                198,\n                7985,\n                6896,\n                2326,\n                31683,\n                72692,\n                61846,\n                369,\n                28990,\n                264,\n                2473,\n                13,\n                151645,\n                198,\n                151644,\n                77091,\n                198\n              ],\n              \"rendered_chat\": \"<|im_start|>system\\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\\n<|im_start|>user\\nWrite exactly three comma-separated verbs for measuring a service.<|im_end|>\\n<|im_start|>assistant\\n\",\n              \"output_tokens\": 24,\n              \"output_token_ids\": [\n                16,\n                13,\n                34104,\n                279,\n                882,\n                4429,\n                311,\n                4583,\n                279,\n                3383,\n                624,\n                17,\n                13,\n                4248,\n                421,\n                279,\n                2473,\n                374,\n                30201,\n                438,\n                3601,\n                624,\n                18,\n                13\n              ],\n              \"output\": \"1. Measure the time taken to complete the task.\\n2. Check if the service is functioning as expected.\\n3.\",\n              \"output_token_pieces\": [\n                \"1\",\n                \".\",\n                \"ĠMeasure\",\n                \"Ġthe\",\n                \"Ġtime\",\n                \"Ġtaken\",\n                \"Ġto\",\n                \"Ġcomplete\",\n                \"Ġthe\",\n                \"Ġtask\",\n                \".Ċ\",\n                \"2\",\n                \".\",\n                \"ĠCheck\",\n                \"Ġif\",\n                \"Ġthe\",\n                \"Ġservice\",\n                \"Ġis\",\n                \"Ġfunctioning\",\n                \"Ġas\",\n                \"Ġexpected\",\n                \".Ċ\",\n                \"3\",\n                \".\"\n              ],\n              \"stop_reason\": \"fixed_length\",\n              \"ttft_s\": 0.016416278000178863,\n              \"wall_s\": 0.46310821899896837,\n              \"post_first_s_per_token\": 0.019421388739077804\n            },\n            {\n              \"prompt_tokens\": 40,\n              \"input_token_ids\": [\n                151644,\n                8948,\n                198,\n                2610,\n                525,\n                1207,\n                16948,\n                11,\n                3465,\n                553,\n                54364,\n                14817,\n                13,\n                1446,\n                525,\n                264,\n                10950,\n                17847,\n                13,\n                151645,\n                198,\n                151644,\n                872,\n                198,\n                7985,\n                6896,\n                2326,\n                31683,\n                72692,\n                61846,\n                369,\n                28990,\n                264,\n                2473,\n                13,\n                151645,\n                198,\n                151644,\n                77091,\n                198\n              ],\n              \"rendered_chat\": \"<|im_start|>system\\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\\n<|im_start|>user\\nWrite exactly three comma-separated verbs for measuring a service.<|im_end|>\\n<|im_start|>assistant\\n\",\n              \"output_tokens\": 24,\n              \"output_token_ids\": [\n                16,\n                13,\n                34104,\n                279,\n                882,\n                4429,\n                311,\n                4583,\n                279,\n                3383,\n                624,\n                17,\n                13,\n                4248,\n                421,\n                279,\n                2473,\n                374,\n                30201,\n                438,\n                3601,\n                624,\n                18,\n                13\n              ],\n              \"output\": \"1. Measure the time taken to complete the task.\\n2. Check if the service is functioning as expected.\\n3.\",\n              \"output_token_pieces\": [\n                \"1\",\n                \".\",\n                \"ĠMeasure\",\n                \"Ġthe\",\n                \"Ġtime\",\n                \"Ġtaken\",\n                \"Ġto\",\n                \"Ġcomplete\",\n                \"Ġthe\",\n                \"Ġtask\",\n                \".Ċ\",\n                \"2\",\n                \".\",\n                \"ĠCheck\",\n                \"Ġif\",\n                \"Ġthe\",\n                \"Ġservice\",\n                \"Ġis\",\n                \"Ġfunctioning\",\n                \"Ġas\",\n                \"Ġexpected\",\n                \".Ċ\",\n                \"3\",\n                \".\"\n              ],\n              \"stop_reason\": \"fixed_length\",\n              \"ttft_s\": 0.038244773999394965,\n              \"wall_s\": 0.3727488490003452,\n              \"post_first_s_per_token\": 0.014543655434823924\n            }\n          ],\n          \"p95_ttft_s\": 0.05025108600057138,\n          \"median_ttft_s\": 0.038244773999394965,\n          \"p95_wall_s\": 0.6263703559998248,\n          \"median_wall_s\": 0.46310821899896837\n        },\n        {\n          \"bucket\": \"reasoning\",\n          \"prompt\": \"A server has 2 GiB free and a new request needs 3 GiB. In one sentence, say whether it fits and why.\",\n          \"quality_sample\": {\n            \"prompt_tokens\": 58,\n            \"input_token_ids\": [\n              151644,\n              8948,\n              198,\n              2610,\n              525,\n              1207,\n              16948,\n              11,\n              3465,\n              553,\n              54364,\n              14817,\n              13,\n              1446,\n              525,\n              264,\n              10950,\n              17847,\n              13,\n              151645,\n              198,\n              151644,\n              872,\n              198,\n              32,\n              3538,\n              702,\n              220,\n              17,\n              15392,\n              33,\n              1910,\n              323,\n              264,\n              501,\n              1681,\n              3880,\n              220,\n              18,\n              15392,\n              33,\n              13,\n              758,\n              825,\n              11652,\n              11,\n              1977,\n              3425,\n              432,\n              18304,\n              323,\n              3170,\n              13,\n              151645,\n              198,\n              151644,\n              77091,\n              198\n            ],\n            \"rendered_chat\": \"<|im_start|>system\\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\\n<|im_start|>user\\nA server has 2 GiB free and a new request needs 3 GiB. In one sentence, say whether it fits and why.<|im_end|>\\n<|im_start|>assistant\\n\",\n            \"output_tokens\": 24,\n            \"output_token_ids\": [\n              785,\n              3538,\n              594,\n              1482,\n              1910,\n              3550,\n              315,\n              220,\n              17,\n              15392,\n              33,\n              374,\n              38313,\n              311,\n              28276,\n              279,\n              11223,\n              220,\n              18,\n              15392,\n              33,\n              315,\n              821,\n              11\n            ],\n            \"output\": \"The server's current free space of 2 GiB is insufficient to accommodate the requested 3 GiB of data,\",\n            \"output_token_pieces\": [\n              \"The\",\n              \"Ġserver\",\n              \"'s\",\n              \"Ġcurrent\",\n              \"Ġfree\",\n              \"Ġspace\",\n              \"Ġof\",\n              \"Ġ\",\n              \"2\",\n              \"ĠGi\",\n              \"B\",\n              \"Ġis\",\n              \"Ġinsufficient\",\n              \"Ġto\",\n              \"Ġaccommodate\",\n              \"Ġthe\",\n              \"Ġrequested\",\n              \"Ġ\",\n              \"3\",\n              \"ĠGi\",\n              \"B\",\n              \"Ġof\",\n              \"Ġdata\",\n              \",\"\n            ],\n            \"stop_reason\": \"length\",\n            \"ttft_s\": 0.016953955999269965,\n            \"wall_s\": 0.5345852840000589,\n            \"post_first_s_per_token\": 0.02250570991307778\n          },\n          \"timing_trials\": [\n            {\n              \"prompt_tokens\": 58,\n              \"input_token_ids\": [\n                151644,\n                8948,\n                198,\n                2610,\n                525,\n                1207,\n                16948,\n                11,\n                3465,\n                553,\n                54364,\n                14817,\n                13,\n                1446,\n                525,\n                264,\n                10950,\n                17847,\n                13,\n                151645,\n                198,\n                151644,\n                872,\n                198,\n                32,\n                3538,\n                702,\n                220,\n                17,\n                15392,\n                33,\n                1910,\n                323,\n                264,\n                501,\n                1681,\n                3880,\n                220,\n                18,\n                15392,\n                33,\n                13,\n                758,\n                825,\n                11652,\n                11,\n                1977,\n                3425,\n                432,\n                18304,\n                323,\n                3170,\n                13,\n                151645,\n                198,\n                151644,\n                77091,\n                198\n              ],\n              \"rendered_chat\": \"<|im_start|>system\\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\\n<|im_start|>user\\nA server has 2 GiB free and a new request needs 3 GiB. In one sentence, say whether it fits and why.<|im_end|>\\n<|im_start|>assistant\\n\",\n              \"output_tokens\": 24,\n              \"output_token_ids\": [\n                785,\n                3538,\n                594,\n                1482,\n                1910,\n                3550,\n                315,\n                220,\n                17,\n                15392,\n                33,\n                374,\n                38313,\n                311,\n                28276,\n                279,\n                11223,\n                220,\n                18,\n                15392,\n                33,\n                315,\n                821,\n                11\n              ],\n              \"output\": \"The server's current free space of 2 GiB is insufficient to accommodate the requested 3 GiB of data,\",\n              \"output_token_pieces\": [\n                \"The\",\n                \"Ġserver\",\n                \"'s\",\n                \"Ġcurrent\",\n                \"Ġfree\",\n                \"Ġspace\",\n                \"Ġof\",\n                \"Ġ\",\n                \"2\",\n                \"ĠGi\",\n                \"B\",\n                \"Ġis\",\n                \"Ġinsufficient\",\n                \"Ġto\",\n                \"Ġaccommodate\",\n                \"Ġthe\",\n                \"Ġrequested\",\n                \"Ġ\",\n                \"3\",\n                \"ĠGi\",\n                \"B\",\n                \"Ġof\",\n                \"Ġdata\",\n                \",\"\n              ],\n              \"stop_reason\": \"fixed_length\",\n              \"ttft_s\": 0.01623754400134203,\n              \"wall_s\": 0.5146201300012763,\n              \"post_first_s_per_token\": 0.02166880808695366\n            },\n            {\n              \"prompt_tokens\": 58,\n              \"input_token_ids\": [\n                151644,\n                8948,\n                198,\n                2610,\n                525,\n                1207,\n                16948,\n                11,\n                3465,\n                553,\n                54364,\n                14817,\n                13,\n                1446,\n                525,\n                264,\n                10950,\n                17847,\n                13,\n                151645,\n                198,\n                151644,\n                872,\n                198,\n                32,\n                3538,\n                702,\n                220,\n                17,\n                15392,\n                33,\n                1910,\n                323,\n                264,\n                501,\n                1681,\n                3880,\n                220,\n                18,\n                15392,\n                33,\n                13,\n                758,\n                825,\n                11652,\n                11,\n                1977,\n                3425,\n                432,\n                18304,\n                323,\n                3170,\n                13,\n                151645,\n                198,\n                151644,\n                77091,\n                198\n              ],\n              \"rendered_chat\": \"<|im_start|>system\\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\\n<|im_start|>user\\nA server has 2 GiB free and a new request needs 3 GiB. In one sentence, say whether it fits and why.<|im_end|>\\n<|im_start|>assistant\\n\",\n              \"output_tokens\": 24,\n              \"output_token_ids\": [\n                785,\n                3538,\n                594,\n                1482,\n                1910,\n                3550,\n                315,\n                220,\n                17,\n                15392,\n                33,\n                374,\n                38313,\n                311,\n                28276,\n                279,\n                11223,\n                220,\n                18,\n                15392,\n                33,\n                315,\n                821,\n                11\n              ],\n              \"output\": \"The server's current free space of 2 GiB is insufficient to accommodate the requested 3 GiB of data,\",\n              \"output_token_pieces\": [\n                \"The\",\n                \"Ġserver\",\n                \"'s\",\n                \"Ġcurrent\",\n                \"Ġfree\",\n                \"Ġspace\",\n                \"Ġof\",\n                \"Ġ\",\n                \"2\",\n                \"ĠGi\",\n                \"B\",\n                \"Ġis\",\n                \"Ġinsufficient\",\n                \"Ġto\",\n                \"Ġaccommodate\",\n                \"Ġthe\",\n                \"Ġrequested\",\n                \"Ġ\",\n                \"3\",\n                \"ĠGi\",\n                \"B\",\n                \"Ġof\",\n                \"Ġdata\",\n                \",\"\n              ],\n              \"stop_reason\": \"fixed_length\",\n              \"ttft_s\": 0.04973485300070024,\n              \"wall_s\": 0.6243948169994837,\n              \"post_first_s_per_token\": 0.024985215826034066\n            },\n            {\n              \"prompt_tokens\": 58,\n              \"input_token_ids\": [\n                151644,\n                8948,\n                198,\n                2610,\n                525,\n                1207,\n                16948,\n                11,\n                3465,\n                553,\n                54364,\n                14817,\n                13,\n                1446,\n                525,\n                264,\n                10950,\n                17847,\n                13,\n                151645,\n                198,\n                151644,\n                872,\n                198,\n                32,\n                3538,\n                702,\n                220,\n                17,\n                15392,\n                33,\n                1910,\n                323,\n                264,\n                501,\n                1681,\n                3880,\n                220,\n                18,\n                15392,\n                33,\n                13,\n                758,\n                825,\n                11652,\n                11,\n                1977,\n                3425,\n                432,\n                18304,\n                323,\n                3170,\n                13,\n                151645,\n                198,\n                151644,\n                77091,\n                198\n              ],\n              \"rendered_chat\": \"<|im_start|>system\\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\\n<|im_start|>user\\nA server has 2 GiB free and a new request needs 3 GiB. In one sentence, say whether it fits and why.<|im_end|>\\n<|im_start|>assistant\\n\",\n              \"output_tokens\": 24,\n              \"output_token_ids\": [\n                785,\n                3538,\n                594,\n                1482,\n                1910,\n                3550,\n                315,\n                220,\n                17,\n                15392,\n                33,\n                374,\n                38313,\n                311,\n                28276,\n                279,\n                11223,\n                220,\n                18,\n                15392,\n                33,\n                315,\n                821,\n                11\n              ],\n              \"output\": \"The server's current free space of 2 GiB is insufficient to accommodate the requested 3 GiB of data,\",\n              \"output_token_pieces\": [\n                \"The\",\n                \"Ġserver\",\n                \"'s\",\n                \"Ġcurrent\",\n                \"Ġfree\",\n                \"Ġspace\",\n                \"Ġof\",\n                \"Ġ\",\n                \"2\",\n                \"ĠGi\",\n                \"B\",\n                \"Ġis\",\n                \"Ġinsufficient\",\n                \"Ġto\",\n                \"Ġaccommodate\",\n                \"Ġthe\",\n                \"Ġrequested\",\n                \"Ġ\",\n                \"3\",\n                \"ĠGi\",\n                \"B\",\n                \"Ġof\",\n                \"Ġdata\",\n                \",\"\n              ],\n              \"stop_reason\": \"fixed_length\",\n              \"ttft_s\": 0.017442587999539683,\n              \"wall_s\": 0.4755077190002339,\n              \"post_first_s_per_token\": 0.01991587526089975\n            }\n          ],\n          \"p95_ttft_s\": 0.04973485300070024,\n          \"median_ttft_s\": 0.017442587999539683,\n          \"p95_wall_s\": 0.6243948169994837,\n          \"median_wall_s\": 0.5146201300012763\n        }\n      ],\n      \"status\": \"complete\",\n      \"baseline_allocated_bytes\": 0,\n      \"eos_token_ids\": [\n        151645,\n        151643\n      ],\n      \"quantized_linear_modules\": 0,\n      \"peak_allocated_bytes\": 1260388352,\n      \"peak_reserved_bytes\": 1277165568,\n      \"post_cleanup_allocated_bytes\": 33554432,\n      \"cleanup_verified\": true\n    },\n    \"nf4_w4a16\": {\n      \"load_s\": 4.197144264999224,\n      \"rows\": [\n        {\n          \"bucket\": \"format\",\n          \"prompt\": \"Reply with exactly one word: blue or orange. What color is the sky on a clear day?\",\n          \"quality_sample\": {\n            \"prompt_tokens\": 49,\n            \"input_token_ids\": [\n              151644,\n              8948,\n              198,\n              2610,\n              525,\n              1207,\n              16948,\n              11,\n              3465,\n              553,\n              54364,\n              14817,\n              13,\n              1446,\n              525,\n              264,\n              10950,\n              17847,\n              13,\n              151645,\n              198,\n              151644,\n              872,\n              198,\n              20841,\n              448,\n              6896,\n              825,\n              3409,\n              25,\n              6303,\n              476,\n              18575,\n              13,\n              3555,\n              1894,\n              374,\n              279,\n              12884,\n              389,\n              264,\n              2797,\n              1899,\n              30,\n              151645,\n              198,\n              151644,\n              77091,\n              198\n            ],\n            \"rendered_chat\": \"<|im_start|>system\\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\\n<|im_start|>user\\nReply with exactly one word: blue or orange. What color is the sky on a clear day?<|im_end|>\\n<|im_start|>assistant\\n\",\n            \"output_tokens\": 3,\n            \"output_token_ids\": [\n              10331,\n              13,\n              151645\n            ],\n            \"output\": \"Blue.\",\n            \"output_token_pieces\": [\n              \"Blue\",\n              \".\",\n              \"<|im_end|>\"\n            ],\n            \"stop_reason\": \"eos\",\n            \"ttft_s\": 0.058474655999816605,\n            \"wall_s\": 0.1026117389992578,\n            \"post_first_s_per_token\": 0.022068541499720595\n          },\n          \"timing_trials\": [\n            {\n              \"prompt_tokens\": 49,\n              \"input_token_ids\": [\n                151644,\n                8948,\n                198,\n                2610,\n                525,\n                1207,\n                16948,\n                11,\n                3465,\n                553,\n                54364,\n                14817,\n                13,\n                1446,\n                525,\n                264,\n                10950,\n                17847,\n                13,\n                151645,\n                198,\n                151644,\n                872,\n                198,\n                20841,\n                448,\n                6896,\n                825,\n                3409,\n                25,\n                6303,\n                476,\n                18575,\n                13,\n                3555,\n                1894,\n                374,\n                279,\n                12884,\n                389,\n                264,\n                2797,\n                1899,\n                30,\n                151645,\n                198,\n                151644,\n                77091,\n                198\n              ],\n              \"rendered_chat\": \"<|im_start|>system\\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\\n<|im_start|>user\\nReply with exactly one word: blue or orange. What color is the sky on a clear day?<|im_end|>\\n<|im_start|>assistant\\n\",\n              \"output_tokens\": 24,\n              \"output_token_ids\": [\n                10331,\n                13,\n                576,\n                12884,\n                7952,\n                6303,\n                1576,\n                315,\n                279,\n                71816,\n                315,\n                39020,\n                438,\n                432,\n                16211,\n                1526,\n                279,\n                16566,\n                13,\n                3197,\n                1052,\n                594,\n                902,\n                9437\n              ],\n              \"output\": \"Blue. The sky appears blue because of the scattering of sunlight as it passes through the atmosphere. When there's no cloud\",\n              \"output_token_pieces\": [\n                \"Blue\",\n                \".\",\n                \"ĠThe\",\n                \"Ġsky\",\n                \"Ġappears\",\n                \"Ġblue\",\n                \"Ġbecause\",\n                \"Ġof\",\n                \"Ġthe\",\n                \"Ġscattering\",\n                \"Ġof\",\n                \"Ġsunlight\",\n                \"Ġas\",\n                \"Ġit\",\n                \"Ġpasses\",\n                \"Ġthrough\",\n                \"Ġthe\",\n                \"Ġatmosphere\",\n                \".\",\n                \"ĠWhen\",\n                \"Ġthere\",\n                \"'s\",\n                \"Ġno\",\n                \"Ġcloud\"\n              ],\n              \"stop_reason\": \"fixed_length\",\n              \"ttft_s\": 0.03110775300046953,\n              \"wall_s\": 0.8063895689992933,\n              \"post_first_s_per_token\": 0.03370790504342712\n            },\n            {\n              \"prompt_tokens\": 49,\n              \"input_token_ids\": [\n                151644,\n                8948,\n                198,\n                2610,\n                525,\n                1207,\n                16948,\n                11,\n                3465,\n                553,\n                54364,\n                14817,\n                13,\n                1446,\n                525,\n                264,\n                10950,\n                17847,\n                13,\n                151645,\n                198,\n                151644,\n                872,\n                198,\n                20841,\n                448,\n                6896,\n                825,\n                3409,\n                25,\n                6303,\n                476,\n                18575,\n                13,\n                3555,\n                1894,\n                374,\n                279,\n                12884,\n                389,\n                264,\n                2797,\n                1899,\n                30,\n                151645,\n                198,\n                151644,\n                77091,\n                198\n              ],\n              \"rendered_chat\": \"<|im_start|>system\\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\\n<|im_start|>user\\nReply with exactly one word: blue or orange. What color is the sky on a clear day?<|im_end|>\\n<|im_start|>assistant\\n\",\n              \"output_tokens\": 24,\n              \"output_token_ids\": [\n                10331,\n                13,\n                576,\n                12884,\n                7952,\n                6303,\n                1576,\n                315,\n                279,\n                71816,\n                315,\n                39020,\n                438,\n                432,\n                16211,\n                1526,\n                279,\n                16566,\n                13,\n                3197,\n                1052,\n                594,\n                902,\n                9437\n              ],\n              \"output\": \"Blue. The sky appears blue because of the scattering of sunlight as it passes through the atmosphere. When there's no cloud\",\n              \"output_token_pieces\": [\n                \"Blue\",\n                \".\",\n                \"ĠThe\",\n                \"Ġsky\",\n                \"Ġappears\",\n                \"Ġblue\",\n                \"Ġbecause\",\n                \"Ġof\",\n                \"Ġthe\",\n                \"Ġscattering\",\n                \"Ġof\",\n                \"Ġsunlight\",\n                \"Ġas\",\n                \"Ġit\",\n                \"Ġpasses\",\n                \"Ġthrough\",\n                \"Ġthe\",\n                \"Ġatmosphere\",\n                \".\",\n                \"ĠWhen\",\n                \"Ġthere\",\n                \"'s\",\n                \"Ġno\",\n                \"Ġcloud\"\n              ],\n              \"stop_reason\": \"fixed_length\",\n              \"ttft_s\": 0.05904763900070975,\n              \"wall_s\": 0.8854131120006059,\n              \"post_first_s_per_token\": 0.035928933608691135\n            },\n            {\n              \"prompt_tokens\": 49,\n              \"input_token_ids\": [\n                151644,\n                8948,\n                198,\n                2610,\n                525,\n                1207,\n                16948,\n                11,\n                3465,\n                553,\n                54364,\n                14817,\n                13,\n                1446,\n                525,\n                264,\n                10950,\n                17847,\n                13,\n                151645,\n                198,\n                151644,\n                872,\n                198,\n                20841,\n                448,\n                6896,\n                825,\n                3409,\n                25,\n                6303,\n                476,\n                18575,\n                13,\n                3555,\n                1894,\n                374,\n                279,\n                12884,\n                389,\n                264,\n                2797,\n                1899,\n                30,\n                151645,\n                198,\n                151644,\n                77091,\n                198\n              ],\n              \"rendered_chat\": \"<|im_start|>system\\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\\n<|im_start|>user\\nReply with exactly one word: blue or orange. What color is the sky on a clear day?<|im_end|>\\n<|im_start|>assistant\\n\",\n              \"output_tokens\": 24,\n              \"output_token_ids\": [\n                10331,\n                13,\n                576,\n                12884,\n                7952,\n                6303,\n                1576,\n                315,\n                279,\n                71816,\n                315,\n                39020,\n                438,\n                432,\n                16211,\n                1526,\n                279,\n                16566,\n                13,\n                3197,\n                1052,\n                594,\n                902,\n                9437\n              ],\n              \"output\": \"Blue. The sky appears blue because of the scattering of sunlight as it passes through the atmosphere. When there's no cloud\",\n              \"output_token_pieces\": [\n                \"Blue\",\n                \".\",\n                \"ĠThe\",\n                \"Ġsky\",\n                \"Ġappears\",\n                \"Ġblue\",\n                \"Ġbecause\",\n                \"Ġof\",\n                \"Ġthe\",\n                \"Ġscattering\",\n                \"Ġof\",\n                \"Ġsunlight\",\n                \"Ġas\",\n                \"Ġit\",\n                \"Ġpasses\",\n                \"Ġthrough\",\n                \"Ġthe\",\n                \"Ġatmosphere\",\n                \".\",\n                \"ĠWhen\",\n                \"Ġthere\",\n                \"'s\",\n                \"Ġno\",\n                \"Ġcloud\"\n              ],\n              \"stop_reason\": \"fixed_length\",\n              \"ttft_s\": 0.041786915999182384,\n              \"wall_s\": 0.8678713449990028,\n              \"post_first_s_per_token\": 0.035916714304340014\n            }\n          ],\n          \"p95_ttft_s\": 0.05904763900070975,\n          \"median_ttft_s\": 0.041786915999182384,\n          \"p95_wall_s\": 0.8854131120006059,\n          \"median_wall_s\": 0.8678713449990028\n        },\n        {\n          \"bucket\": \"arithmetic\",\n          \"prompt\": \"Compute 17 + 28. Give only the integer.\",\n          \"quality_sample\": {\n            \"prompt_tokens\": 43,\n            \"input_token_ids\": [\n              151644,\n              8948,\n              198,\n              2610,\n              525,\n              1207,\n              16948,\n              11,\n              3465,\n              553,\n              54364,\n              14817,\n              13,\n              1446,\n              525,\n              264,\n              10950,\n              17847,\n              13,\n              151645,\n              198,\n              151644,\n              872,\n              198,\n              46254,\n              220,\n              16,\n              22,\n              488,\n              220,\n              17,\n              23,\n              13,\n              20678,\n              1172,\n              279,\n              7546,\n              13,\n              151645,\n              198,\n              151644,\n              77091,\n              198\n            ],\n            \"rendered_chat\": \"<|im_start|>system\\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\\n<|im_start|>user\\nCompute 17 + 28. Give only the integer.<|im_end|>\\n<|im_start|>assistant\\n\",\n            \"output_tokens\": 3,\n            \"output_token_ids\": [\n              19,\n              20,\n              151645\n            ],\n            \"output\": \"45\",\n            \"output_token_pieces\": [\n              \"4\",\n              \"5\",\n              \"<|im_end|>\"\n            ],\n            \"stop_reason\": \"eos\",\n            \"ttft_s\": 0.039785569000741816,\n            \"wall_s\": 0.07856106299914245,\n            \"post_first_s_per_token\": 0.019387746999200317\n          },\n          \"timing_trials\": [\n            {\n              \"prompt_tokens\": 43,\n              \"input_token_ids\": [\n                151644,\n                8948,\n                198,\n                2610,\n                525,\n                1207,\n                16948,\n                11,\n                3465,\n                553,\n                54364,\n                14817,\n                13,\n                1446,\n                525,\n                264,\n                10950,\n                17847,\n                13,\n                151645,\n                198,\n                151644,\n                872,\n                198,\n                46254,\n                220,\n                16,\n                22,\n                488,\n                220,\n                17,\n                23,\n                13,\n                20678,\n                1172,\n                279,\n                7546,\n                13,\n                151645,\n                198,\n                151644,\n                77091,\n                198\n              ],\n              \"rendered_chat\": \"<|im_start|>system\\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\\n<|im_start|>user\\nCompute 17 + 28. Give only the integer.<|im_end|>\\n<|im_start|>assistant\\n\",\n              \"output_tokens\": 24,\n              \"output_token_ids\": [\n                19,\n                20,\n                13,\n                15,\n                15,\n                15,\n                15,\n                15,\n                15,\n                15,\n                15,\n                15,\n                15,\n                15,\n                15,\n                15,\n                15,\n                18,\n                320,\n                43991,\n                311,\n                458,\n                7546,\n                8\n              ],\n              \"output\": \"45.000000000000003 (rounded to an integer)\",\n              \"output_token_pieces\": [\n                \"4\",\n                \"5\",\n                \".\",\n                \"0\",\n                \"0\",\n                \"0\",\n                \"0\",\n                \"0\",\n                \"0\",\n                \"0\",\n                \"0\",\n                \"0\",\n                \"0\",\n                \"0\",\n                \"0\",\n                \"0\",\n                \"0\",\n                \"3\",\n                \"Ġ(\",\n                \"rounded\",\n                \"Ġto\",\n                \"Ġan\",\n                \"Ġinteger\",\n                \")\"\n              ],\n              \"stop_reason\": \"fixed_length\",\n              \"ttft_s\": 0.02536492199942586,\n              \"wall_s\": 0.6565068289983174,\n              \"post_first_s_per_token\": 0.027440952478212675\n            },\n            {\n              \"prompt_tokens\": 43,\n              \"input_token_ids\": [\n                151644,\n                8948,\n                198,\n                2610,\n                525,\n                1207,\n                16948,\n                11,\n                3465,\n                553,\n                54364,\n                14817,\n                13,\n                1446,\n                525,\n                264,\n                10950,\n                17847,\n                13,\n                151645,\n                198,\n                151644,\n                872,\n                198,\n                46254,\n                220,\n                16,\n                22,\n                488,\n                220,\n                17,\n                23,\n                13,\n                20678,\n                1172,\n                279,\n                7546,\n                13,\n                151645,\n                198,\n                151644,\n                77091,\n                198\n              ],\n              \"rendered_chat\": \"<|im_start|>system\\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\\n<|im_start|>user\\nCompute 17 + 28. Give only the integer.<|im_end|>\\n<|im_start|>assistant\\n\",\n              \"output_tokens\": 24,\n              \"output_token_ids\": [\n                19,\n                20,\n                13,\n                15,\n                15,\n                15,\n                15,\n                15,\n                15,\n                15,\n                15,\n                15,\n                15,\n                15,\n                15,\n                15,\n                15,\n                18,\n                320,\n                43991,\n                311,\n                458,\n                7546,\n                8\n              ],\n              \"output\": \"45.000000000000003 (rounded to an integer)\",\n              \"output_token_pieces\": [\n                \"4\",\n                \"5\",\n                \".\",\n                \"0\",\n                \"0\",\n                \"0\",\n                \"0\",\n                \"0\",\n                \"0\",\n                \"0\",\n                \"0\",\n                \"0\",\n                \"0\",\n                \"0\",\n                \"0\",\n                \"0\",\n                \"0\",\n                \"3\",\n                \"Ġ(\",\n                \"rounded\",\n                \"Ġto\",\n                \"Ġan\",\n                \"Ġinteger\",\n                \")\"\n              ],\n              \"stop_reason\": \"fixed_length\",\n              \"ttft_s\": 0.025343840999994427,\n              \"wall_s\": 0.6756680739999865,\n              \"post_first_s_per_token\": 0.02827496665217357\n            },\n            {\n              \"prompt_tokens\": 43,\n              \"input_token_ids\": [\n                151644,\n                8948,\n                198,\n                2610,\n                525,\n                1207,\n                16948,\n                11,\n                3465,\n                553,\n                54364,\n                14817,\n                13,\n                1446,\n                525,\n                264,\n                10950,\n                17847,\n                13,\n                151645,\n                198,\n                151644,\n                872,\n                198,\n                46254,\n                220,\n                16,\n                22,\n                488,\n                220,\n                17,\n                23,\n                13,\n                20678,\n                1172,\n                279,\n                7546,\n                13,\n                151645,\n                198,\n                151644,\n                77091,\n                198\n              ],\n              \"rendered_chat\": \"<|im_start|>system\\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\\n<|im_start|>user\\nCompute 17 + 28. Give only the integer.<|im_end|>\\n<|im_start|>assistant\\n\",\n              \"output_tokens\": 24,\n              \"output_token_ids\": [\n                19,\n                20,\n                13,\n                15,\n                15,\n                15,\n                15,\n                15,\n                15,\n                15,\n                15,\n                15,\n                15,\n                15,\n                15,\n                15,\n                15,\n                18,\n                320,\n                43991,\n                311,\n                458,\n                7546,\n                8\n              ],\n              \"output\": \"45.000000000000003 (rounded to an integer)\",\n              \"output_token_pieces\": [\n                \"4\",\n                \"5\",\n                \".\",\n                \"0\",\n                \"0\",\n                \"0\",\n                \"0\",\n                \"0\",\n                \"0\",\n                \"0\",\n                \"0\",\n                \"0\",\n                \"0\",\n                \"0\",\n                \"0\",\n                \"0\",\n                \"0\",\n                \"3\",\n                \"Ġ(\",\n                \"rounded\",\n                \"Ġto\",\n                \"Ġan\",\n                \"Ġinteger\",\n                \")\"\n              ],\n              \"stop_reason\": \"fixed_length\",\n              \"ttft_s\": 0.042260009999154136,\n              \"wall_s\": 0.7717349049999029,\n              \"post_first_s_per_token\": 0.031716299782641254\n            }\n          ],\n          \"p95_ttft_s\": 0.042260009999154136,\n          \"median_ttft_s\": 0.02536492199942586,\n          \"p95_wall_s\": 0.7717349049999029,\n          \"median_wall_s\": 0.6756680739999865\n        },\n        {\n          \"bucket\": \"instruction\",\n          \"prompt\": \"Write exactly three comma-separated verbs for measuring a service.\",\n          \"quality_sample\": {\n            \"prompt_tokens\": 40,\n            \"input_token_ids\": [\n              151644,\n              8948,\n              198,\n              2610,\n              525,\n              1207,\n              16948,\n              11,\n              3465,\n              553,\n              54364,\n              14817,\n              13,\n              1446,\n              525,\n              264,\n              10950,\n              17847,\n              13,\n              151645,\n              198,\n              151644,\n              872,\n              198,\n              7985,\n              6896,\n              2326,\n              31683,\n              72692,\n              61846,\n              369,\n              28990,\n              264,\n              2473,\n              13,\n              151645,\n              198,\n              151644,\n              77091,\n              198\n            ],\n            \"rendered_chat\": \"<|im_start|>system\\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\\n<|im_start|>user\\nWrite exactly three comma-separated verbs for measuring a service.<|im_end|>\\n<|im_start|>assistant\\n\",\n            \"output_tokens\": 12,\n            \"output_token_ids\": [\n              16,\n              13,\n              52932,\n              198,\n              17,\n              13,\n              39565,\n              198,\n              18,\n              13,\n              24498,\n              151645\n            ],\n            \"output\": \"1. Serve\\n2. Provide\\n3. Offer\",\n            \"output_token_pieces\": [\n              \"1\",\n              \".\",\n              \"ĠServe\",\n              \"Ċ\",\n              \"2\",\n              \".\",\n              \"ĠProvide\",\n              \"Ċ\",\n              \"3\",\n              \".\",\n              \"ĠOffer\",\n              \"<|im_end|>\"\n            ],\n            \"stop_reason\": \"eos\",\n            \"ttft_s\": 0.024818369000058738,\n            \"wall_s\": 0.398327941999014,\n            \"post_first_s_per_token\": 0.03395541572717775\n          },\n          \"timing_trials\": [\n            {\n              \"prompt_tokens\": 40,\n              \"input_token_ids\": [\n                151644,\n                8948,\n                198,\n                2610,\n                525,\n                1207,\n                16948,\n                11,\n                3465,\n                553,\n                54364,\n                14817,\n                13,\n                1446,\n                525,\n                264,\n                10950,\n                17847,\n                13,\n                151645,\n                198,\n                151644,\n                872,\n                198,\n                7985,\n                6896,\n                2326,\n                31683,\n                72692,\n                61846,\n                369,\n                28990,\n                264,\n                2473,\n                13,\n                151645,\n                198,\n                151644,\n                77091,\n                198\n              ],\n              \"rendered_chat\": \"<|im_start|>system\\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\\n<|im_start|>user\\nWrite exactly three comma-separated verbs for measuring a service.<|im_end|>\\n<|im_start|>assistant\\n\",\n              \"output_tokens\": 24,\n              \"output_token_ids\": [\n                16,\n                13,\n                52932,\n                198,\n                17,\n                13,\n                39565,\n                198,\n                18,\n                13,\n                24498,\n                198,\n                9485,\n                61846,\n                646,\n                387,\n                1483,\n                311,\n                7512,\n                279,\n                1160,\n                315,\n                8241,\n                476\n              ],\n              \"output\": \"1. Serve\\n2. Provide\\n3. Offer\\nThese verbs can be used to describe the act of providing or\",\n              \"output_token_pieces\": [\n                \"1\",\n                \".\",\n                \"ĠServe\",\n                \"Ċ\",\n                \"2\",\n                \".\",\n                \"ĠProvide\",\n                \"Ċ\",\n                \"3\",\n                \".\",\n                \"ĠOffer\",\n                \"Ċ\",\n                \"These\",\n                \"Ġverbs\",\n                \"Ġcan\",\n                \"Ġbe\",\n                \"Ġused\",\n                \"Ġto\",\n                \"Ġdescribe\",\n                \"Ġthe\",\n                \"Ġact\",\n                \"Ġof\",\n                \"Ġproviding\",\n                \"Ġor\"\n              ],\n              \"stop_reason\": \"fixed_length\",\n              \"ttft_s\": 0.026642100998287788,\n              \"wall_s\": 0.637539431998448,\n              \"post_first_s_per_token\": 0.026560753521746094\n            },\n            {\n              \"prompt_tokens\": 40,\n              \"input_token_ids\": [\n                151644,\n                8948,\n                198,\n                2610,\n                525,\n                1207,\n                16948,\n                11,\n                3465,\n                553,\n                54364,\n                14817,\n                13,\n                1446,\n                525,\n                264,\n                10950,\n                17847,\n                13,\n                151645,\n                198,\n                151644,\n                872,\n                198,\n                7985,\n                6896,\n                2326,\n                31683,\n                72692,\n                61846,\n                369,\n                28990,\n                264,\n                2473,\n                13,\n                151645,\n                198,\n                151644,\n                77091,\n                198\n              ],\n              \"rendered_chat\": \"<|im_start|>system\\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\\n<|im_start|>user\\nWrite exactly three comma-separated verbs for measuring a service.<|im_end|>\\n<|im_start|>assistant\\n\",\n              \"output_tokens\": 24,\n              \"output_token_ids\": [\n                16,\n                13,\n                52932,\n                198,\n                17,\n                13,\n                39565,\n                198,\n                18,\n                13,\n                24498,\n                198,\n                9485,\n                61846,\n                646,\n                387,\n                1483,\n                311,\n                7512,\n                279,\n                1160,\n                315,\n                8241,\n                476\n              ],\n              \"output\": \"1. Serve\\n2. Provide\\n3. Offer\\nThese verbs can be used to describe the act of providing or\",\n              \"output_token_pieces\": [\n                \"1\",\n                \".\",\n                \"ĠServe\",\n                \"Ċ\",\n                \"2\",\n                \".\",\n                \"ĠProvide\",\n                \"Ċ\",\n                \"3\",\n                \".\",\n                \"ĠOffer\",\n                \"Ċ\",\n                \"These\",\n                \"Ġverbs\",\n                \"Ġcan\",\n                \"Ġbe\",\n                \"Ġused\",\n                \"Ġto\",\n                \"Ġdescribe\",\n                \"Ġthe\",\n                \"Ġact\",\n                \"Ġof\",\n                \"Ġproviding\",\n                \"Ġor\"\n              ],\n              \"stop_reason\": \"fixed_length\",\n              \"ttft_s\": 0.04989953299991612,\n              \"wall_s\": 0.5735576050010422,\n              \"post_first_s_per_token\": 0.022767742260918527\n            },\n            {\n              \"prompt_tokens\": 40,\n              \"input_token_ids\": [\n                151644,\n                8948,\n                198,\n                2610,\n                525,\n                1207,\n                16948,\n                11,\n                3465,\n                553,\n                54364,\n                14817,\n                13,\n                1446,\n                525,\n                264,\n                10950,\n                17847,\n                13,\n                151645,\n                198,\n                151644,\n                872,\n                198,\n                7985,\n                6896,\n                2326,\n                31683,\n                72692,\n                61846,\n                369,\n                28990,\n                264,\n                2473,\n                13,\n                151645,\n                198,\n                151644,\n                77091,\n                198\n              ],\n              \"rendered_chat\": \"<|im_start|>system\\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\\n<|im_start|>user\\nWrite exactly three comma-separated verbs for measuring a service.<|im_end|>\\n<|im_start|>assistant\\n\",\n              \"output_tokens\": 24,\n              \"output_token_ids\": [\n                16,\n                13,\n                52932,\n                198,\n                17,\n                13,\n                39565,\n                198,\n                18,\n                13,\n                24498,\n                198,\n                9485,\n                61846,\n                646,\n                387,\n                1483,\n                311,\n                7512,\n                279,\n                1160,\n                315,\n                8241,\n                476\n              ],\n              \"output\": \"1. Serve\\n2. Provide\\n3. Offer\\nThese verbs can be used to describe the act of providing or\",\n              \"output_token_pieces\": [\n                \"1\",\n                \".\",\n                \"ĠServe\",\n                \"Ċ\",\n                \"2\",\n                \".\",\n                \"ĠProvide\",\n                \"Ċ\",\n                \"3\",\n                \".\",\n                \"ĠOffer\",\n                \"Ċ\",\n                \"These\",\n                \"Ġverbs\",\n                \"Ġcan\",\n                \"Ġbe\",\n                \"Ġused\",\n                \"Ġto\",\n                \"Ġdescribe\",\n                \"Ġthe\",\n                \"Ġact\",\n                \"Ġof\",\n                \"Ġproviding\",\n                \"Ġor\"\n              ],\n              \"stop_reason\": \"fixed_length\",\n              \"ttft_s\": 0.03874448399983521,\n              \"wall_s\": 0.7476808689989412,\n              \"post_first_s_per_token\": 0.03082332108691765\n            }\n          ],\n          \"p95_ttft_s\": 0.04989953299991612,\n          \"median_ttft_s\": 0.03874448399983521,\n          \"p95_wall_s\": 0.7476808689989412,\n          \"median_wall_s\": 0.637539431998448\n        },\n        {\n          \"bucket\": \"reasoning\",\n          \"prompt\": \"A server has 2 GiB free and a new request needs 3 GiB. In one sentence, say whether it fits and why.\",\n          \"quality_sample\": {\n            \"prompt_tokens\": 58,\n            \"input_token_ids\": [\n              151644,\n              8948,\n              198,\n              2610,\n              525,\n              1207,\n              16948,\n              11,\n              3465,\n              553,\n              54364,\n              14817,\n              13,\n              1446,\n              525,\n              264,\n              10950,\n              17847,\n              13,\n              151645,\n              198,\n              151644,\n              872,\n              198,\n              32,\n              3538,\n              702,\n              220,\n              17,\n              15392,\n              33,\n              1910,\n              323,\n              264,\n              501,\n              1681,\n              3880,\n              220,\n              18,\n              15392,\n              33,\n              13,\n              758,\n              825,\n              11652,\n              11,\n              1977,\n              3425,\n              432,\n              18304,\n              323,\n              3170,\n              13,\n              151645,\n              198,\n              151644,\n              77091,\n              198\n            ],\n            \"rendered_chat\": \"<|im_start|>system\\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\\n<|im_start|>user\\nA server has 2 GiB free and a new request needs 3 GiB. In one sentence, say whether it fits and why.<|im_end|>\\n<|im_start|>assistant\\n\",\n            \"output_tokens\": 24,\n            \"output_token_ids\": [\n              2132,\n              1558,\n              537,\n              4946,\n              1576,\n              279,\n              3538,\n              702,\n              220,\n              17,\n              15392,\n              33,\n              315,\n              1910,\n              3550,\n              11,\n              892,\n              374,\n              38313,\n              311,\n              28276,\n              279,\n              11223,\n              220\n            ],\n            \"output\": \"It does not fit because the server has 2 GiB of free space, which is insufficient to accommodate the requested \",\n            \"output_token_pieces\": [\n              \"It\",\n              \"Ġdoes\",\n              \"Ġnot\",\n              \"Ġfit\",\n              \"Ġbecause\",\n              \"Ġthe\",\n              \"Ġserver\",\n              \"Ġhas\",\n              \"Ġ\",\n              \"2\",\n              \"ĠGi\",\n              \"B\",\n              \"Ġof\",\n              \"Ġfree\",\n              \"Ġspace\",\n              \",\",\n              \"Ġwhich\",\n              \"Ġis\",\n              \"Ġinsufficient\",\n              \"Ġto\",\n              \"Ġaccommodate\",\n              \"Ġthe\",\n              \"Ġrequested\",\n              \"Ġ\"\n            ],\n            \"stop_reason\": \"length\",\n            \"ttft_s\": 0.05997879399910744,\n            \"wall_s\": 0.8479015969987813,\n            \"post_first_s_per_token\": 0.034257513173898864\n          },\n          \"timing_trials\": [\n            {\n              \"prompt_tokens\": 58,\n              \"input_token_ids\": [\n                151644,\n                8948,\n                198,\n                2610,\n                525,\n                1207,\n                16948,\n                11,\n                3465,\n                553,\n                54364,\n                14817,\n                13,\n                1446,\n                525,\n                264,\n                10950,\n                17847,\n                13,\n                151645,\n                198,\n                151644,\n                872,\n                198,\n                32,\n                3538,\n                702,\n                220,\n                17,\n                15392,\n                33,\n                1910,\n                323,\n                264,\n                501,\n                1681,\n                3880,\n                220,\n                18,\n                15392,\n                33,\n                13,\n                758,\n                825,\n                11652,\n                11,\n                1977,\n                3425,\n                432,\n                18304,\n                323,\n                3170,\n                13,\n                151645,\n                198,\n                151644,\n                77091,\n                198\n              ],\n              \"rendered_chat\": \"<|im_start|>system\\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\\n<|im_start|>user\\nA server has 2 GiB free and a new request needs 3 GiB. In one sentence, say whether it fits and why.<|im_end|>\\n<|im_start|>assistant\\n\",\n              \"output_tokens\": 24,\n              \"output_token_ids\": [\n                2132,\n                1558,\n                537,\n                4946,\n                1576,\n                279,\n                3538,\n                702,\n                220,\n                17,\n                15392,\n                33,\n                315,\n                1910,\n                3550,\n                11,\n                892,\n                374,\n                38313,\n                311,\n                28276,\n                279,\n                11223,\n                220\n              ],\n              \"output\": \"It does not fit because the server has 2 GiB of free space, which is insufficient to accommodate the requested \",\n              \"output_token_pieces\": [\n                \"It\",\n                \"Ġdoes\",\n                \"Ġnot\",\n                \"Ġfit\",\n                \"Ġbecause\",\n                \"Ġthe\",\n                \"Ġserver\",\n                \"Ġhas\",\n                \"Ġ\",\n                \"2\",\n                \"ĠGi\",\n                \"B\",\n                \"Ġof\",\n                \"Ġfree\",\n                \"Ġspace\",\n                \",\",\n                \"Ġwhich\",\n                \"Ġis\",\n                \"Ġinsufficient\",\n                \"Ġto\",\n                \"Ġaccommodate\",\n                \"Ġthe\",\n                \"Ġrequested\",\n                \"Ġ\"\n              ],\n              \"stop_reason\": \"fixed_length\",\n              \"ttft_s\": 0.05709481399935612,\n              \"wall_s\": 1.0016033859992604,\n              \"post_first_s_per_token\": 0.04106559008695236\n            },\n            {\n              \"prompt_tokens\": 58,\n              \"input_token_ids\": [\n                151644,\n                8948,\n                198,\n                2610,\n                525,\n                1207,\n                16948,\n                11,\n                3465,\n                553,\n                54364,\n                14817,\n                13,\n                1446,\n                525,\n                264,\n                10950,\n                17847,\n                13,\n                151645,\n                198,\n                151644,\n                872,\n                198,\n                32,\n                3538,\n                702,\n                220,\n                17,\n                15392,\n                33,\n                1910,\n                323,\n                264,\n                501,\n                1681,\n                3880,\n                220,\n                18,\n                15392,\n                33,\n                13,\n                758,\n                825,\n                11652,\n                11,\n                1977,\n                3425,\n                432,\n                18304,\n                323,\n                3170,\n                13,\n                151645,\n                198,\n                151644,\n                77091,\n                198\n              ],\n              \"rendered_chat\": \"<|im_start|>system\\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\\n<|im_start|>user\\nA server has 2 GiB free and a new request needs 3 GiB. In one sentence, say whether it fits and why.<|im_end|>\\n<|im_start|>assistant\\n\",\n              \"output_tokens\": 24,\n              \"output_token_ids\": [\n                2132,\n                1558,\n                537,\n                4946,\n                1576,\n                279,\n                3538,\n                702,\n                220,\n                17,\n                15392,\n                33,\n                315,\n                1910,\n                3550,\n                11,\n                892,\n                374,\n                38313,\n                311,\n                28276,\n                279,\n                11223,\n                220\n              ],\n              \"output\": \"It does not fit because the server has 2 GiB of free space, which is insufficient to accommodate the requested \",\n              \"output_token_pieces\": [\n                \"It\",\n                \"Ġdoes\",\n                \"Ġnot\",\n                \"Ġfit\",\n                \"Ġbecause\",\n                \"Ġthe\",\n                \"Ġserver\",\n                \"Ġhas\",\n                \"Ġ\",\n                \"2\",\n                \"ĠGi\",\n                \"B\",\n                \"Ġof\",\n                \"Ġfree\",\n                \"Ġspace\",\n                \",\",\n                \"Ġwhich\",\n                \"Ġis\",\n                \"Ġinsufficient\",\n                \"Ġto\",\n                \"Ġaccommodate\",\n                \"Ġthe\",\n                \"Ġrequested\",\n                \"Ġ\"\n              ],\n              \"stop_reason\": \"fixed_length\",\n              \"ttft_s\": 0.030300465999971493,\n              \"wall_s\": 0.90360615300051,\n              \"post_first_s_per_token\": 0.03796981247828428\n            },\n            {\n              \"prompt_tokens\": 58,\n              \"input_token_ids\": [\n                151644,\n                8948,\n                198,\n                2610,\n                525,\n                1207,\n                16948,\n                11,\n                3465,\n                553,\n                54364,\n                14817,\n                13,\n                1446,\n                525,\n                264,\n                10950,\n                17847,\n                13,\n                151645,\n                198,\n                151644,\n                872,\n                198,\n                32,\n                3538,\n                702,\n                220,\n                17,\n                15392,\n                33,\n                1910,\n                323,\n                264,\n                501,\n                1681,\n                3880,\n                220,\n                18,\n                15392,\n                33,\n                13,\n                758,\n                825,\n                11652,\n                11,\n                1977,\n                3425,\n                432,\n                18304,\n                323,\n                3170,\n                13,\n                151645,\n                198,\n                151644,\n                77091,\n                198\n              ],\n              \"rendered_chat\": \"<|im_start|>system\\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\\n<|im_start|>user\\nA server has 2 GiB free and a new request needs 3 GiB. In one sentence, say whether it fits and why.<|im_end|>\\n<|im_start|>assistant\\n\",\n              \"output_tokens\": 24,\n              \"output_token_ids\": [\n                2132,\n                1558,\n                537,\n                4946,\n                1576,\n                279,\n                3538,\n                702,\n                220,\n                17,\n                15392,\n                33,\n                315,\n                1910,\n                3550,\n                11,\n                892,\n                374,\n                38313,\n                311,\n                28276,\n                279,\n                11223,\n                220\n              ],\n              \"output\": \"It does not fit because the server has 2 GiB of free space, which is insufficient to accommodate the requested \",\n              \"output_token_pieces\": [\n                \"It\",\n                \"Ġdoes\",\n                \"Ġnot\",\n                \"Ġfit\",\n                \"Ġbecause\",\n                \"Ġthe\",\n                \"Ġserver\",\n                \"Ġhas\",\n                \"Ġ\",\n                \"2\",\n                \"ĠGi\",\n                \"B\",\n                \"Ġof\",\n                \"Ġfree\",\n                \"Ġspace\",\n                \",\",\n                \"Ġwhich\",\n                \"Ġis\",\n                \"Ġinsufficient\",\n                \"Ġto\",\n                \"Ġaccommodate\",\n                \"Ġthe\",\n                \"Ġrequested\",\n                \"Ġ\"\n              ],\n              \"stop_reason\": \"fixed_length\",\n              \"ttft_s\": 0.056263073998707114,\n              \"wall_s\": 0.9429308929993567,\n              \"post_first_s_per_token\": 0.03855077473915868\n            }\n          ],\n          \"p95_ttft_s\": 0.05709481399935612,\n          \"median_ttft_s\": 0.056263073998707114,\n          \"p95_wall_s\": 1.0016033859992604,\n          \"median_wall_s\": 0.9429308929993567\n        }\n      ],\n      \"status\": \"complete\",\n      \"baseline_allocated_bytes\": 33554432,\n      \"eos_token_ids\": [\n        151645,\n        151643\n      ],\n      \"quantized_linear_modules\": 168,\n      \"peak_allocated_bytes\": 663721984,\n      \"peak_reserved_bytes\": 1310720000,\n      \"post_cleanup_allocated_bytes\": 33554432,\n      \"cleanup_verified\": true\n    }\n  },\n  \"resource_samples\": [\n    {\n      \"available_ram_gib\": 68.9375228881836,\n      \"device_used_mib\": 0\n    },\n    {\n      \"available_ram_gib\": 67.92237091064453,\n      \"device_used_mib\": 1478\n    },\n    {\n      \"available_ram_gib\": 67.88650894165039,\n      \"device_used_mib\": 1478\n    }\n  ],\n  \"clock\": \"synchronized generate wall; first generated-ID streamer callback; post-first proxy includes return overhead, not last-emission interval; not HTTP or pure GPU time\",\n  \"quantization\": {\n    \"quant_type\": \"nf4\",\n    \"compute_dtype\": \"float16\",\n    \"double_quant\": false\n  },\n  \"limitations\": [\n    \"four-prompt smoke, not quality benchmark\",\n    \"fixed format order, three trials, uncontrolled timing variance\",\n    \"allocated/reserved are PyTorch allocator counters, not complete process/device memory\"\n  ],\n  \"utc_finished\": \"2026-10-01T22:02:29.744975+00:00\"\n}\n")
CASE=1  # format=0, arithmetic=1, instruction=2, reasoning=3
assert CASE in range(4)
paired=[]
for label in ('fp16','nf4_w4a16'):
    row=RECORDED_QUANTIZATION['formats'][label]['rows'][CASE]
    sample=row['quality_sample']; paired.append(sample['input_token_ids'])
    print('\nRECORDED',label,'literal prompt:',row['prompt'])
    print('Rendered chat:',repr(sample['rendered_chat']))
    print('Input IDs/count:',sample['input_token_ids'],sample['prompt_tokens'])
    print('Natural output:',repr(sample['output']),'IDs:',sample['output_token_ids'],'stop:',sample['stop_reason'])
    for index,trial in enumerate(row['timing_trials']):
        print('Fixed24 trial',index,'first-ID s:',trial['ttft_s'],'generate wall s:',trial['wall_s'],'post-first s:',trial['post_first_s_per_token'])
assert paired[0]==paired[1]
print('Recorded teaching smoke; not your own GPU evidence or a speed guarantee.')


## 2. GPU readiness, safety and persistent cache

Write your predictions for payload, allocator peak and output quality before running. Use one process and an idle GPU. This canonical implementation requires Linux/WSL, at least **24 GiB available host RAM**, and no more than 12 GiB device use. These gates are not an OS sandbox: instructor captures additionally verify a 24 GiB/no-swap cgroup and wall timeout. Do not lower a gate to force a run through; move to an adequately provisioned CUDA machine.

Set LSM_MODEL_CACHE to persistent storage. Existing exact packages and pinned snapshots are reused; HF_HUB_OFFLINE=1 fails closed if anything is missing. A fresh output directory never clears model caches. The next install cell skips matching versions; restart the kernel if you already imported incompatible packages.


In [ ]:
import subprocess,sys
from importlib.metadata import version,PackageNotFoundError
pinned={'transformers':'4.51.3','accelerate':'1.6.0','bitsandbytes':'0.50.2','huggingface_hub':'0.30.2','safetensors':'0.5.3'}
for name,wanted in pinned.items():
    try: matches=version(name)==wanted
    except PackageNotFoundError: matches=False
    if not matches: subprocess.check_call([sys.executable,'-m','pip','install',name+'=='+wanted])


In [ ]:
lab_namespace={'__name__':'topic03_student_lab'}
exec("\"\"\"Canonical bounded paired smoke. Import is CPU-only; GPU work starts at run_lab().\"\"\"\nimport gc\nimport json\nimport math\nimport os\nimport statistics\nimport subprocess\nimport time\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nMODEL_ID='Qwen/Qwen2.5-0.5B-Instruct'\nREVISION='7ae557604adf67be50417f59c2c2f167def9a775'\nPROTOCOL='topic-03-paired-smoke-v3'\nTRIALS, MAX_NEW=3,24\nPROMPTS=[\n    dict(bucket='format',text='Reply with exactly one word: blue or orange. What color is the sky on a clear day?'),\n    dict(bucket='arithmetic',text='Compute 17 + 28. Give only the integer.'),\n    dict(bucket='instruction',text='Write exactly three comma-separated verbs for measuring a service.'),\n    dict(bucket='reasoning',text='A server has 2 GiB free and a new request needs 3 GiB. In one sentence, say whether it fits and why.'),\n]\n\n\ndef checkpoint(path,data):\n    path=Path(path); temporary=path.with_suffix(path.suffix+'.tmp')\n    temporary.write_text(json.dumps(data,indent=2,ensure_ascii=False,allow_nan=False)+'\\n',encoding='utf-8')\n    temporary.replace(path)\n\n\ndef resource_gate(require_idle=False):\n    mem=Path('/proc/meminfo')\n    if not mem.exists(): raise RuntimeError('Linux/WSL RAM gate unavailable; use supported CUDA runtime')\n    available=int(next(line.split()[1] for line in mem.read_text().splitlines() if line.startswith('MemAvailable:')))*1024\n    used=[int(x) for x in subprocess.check_output(['nvidia-smi','--query-gpu=memory.used','--format=csv,noheader,nounits'],text=True,timeout=10).split()]\n    if available < 24*2**30 or not used or max(used)>12288: raise RuntimeError('STOP: RAM/GPU resource gate')\n    if require_idle:\n        active=subprocess.check_output(['nvidia-smi','--query-compute-apps=pid','--format=csv,noheader,nounits'],text=True,timeout=10).strip()\n        if active: raise RuntimeError('STOP: GPU compute process already active')\n    return dict(available_ram_gib=available/2**30,device_used_mib=max(used))\n\n\ndef resolve_snapshot():\n    from huggingface_hub import snapshot_download\n    cache=Path(os.environ.get('LSM_MODEL_CACHE',Path.cwd()/'.cache/models')).expanduser()\n    try: snapshot=snapshot_download(MODEL_ID,revision=REVISION,cache_dir=str(cache),local_files_only=True)\n    except FileNotFoundError:\n        if os.environ.get('HF_HUB_OFFLINE')=='1': raise RuntimeError('Pinned offline snapshot missing; do not redownload')\n        snapshot=snapshot_download(MODEL_ID,revision=REVISION,cache_dir=str(cache))\n    return Path(snapshot)\n\n\ndef run_lab(snapshot,output_path):\n    import torch,transformers,bitsandbytes\n    from transformers import AutoModelForCausalLM,AutoTokenizer,BitsAndBytesConfig\n    from transformers.generation.streamers import BaseStreamer\n    if transformers.__version__!='4.51.3' or bitsandbytes.__version__!='0.50.2':\n        raise RuntimeError('Pinned Transformers/bitsandbytes versions required')\n    output_path=Path(output_path)\n    if output_path.exists(): raise FileExistsError('Preserve prior evidence: choose a fresh output path')\n    output_path.parent.mkdir(parents=True,exist_ok=True)\n    preflight=resource_gate(require_idle=True)\n    if not torch.cuda.is_available(): raise RuntimeError('CUDA required; recorded replay is not execution evidence')\n    tokenizer=AutoTokenizer.from_pretrained(str(snapshot),local_files_only=True)\n    class TokenClock(BaseStreamer):\n        def __init__(self): self.seen_prompt=False; self.first=None\n        def put(self,value):\n            if not self.seen_prompt: self.seen_prompt=True; return\n            # Same callback overhead in both treatments; not HTTP first-content time.\n            torch.cuda.synchronize()\n            if self.first is None: self.first=time.perf_counter()\n        def end(self): pass\n    def one(model,case,force):\n        messages=[dict(role='user',content=case['text'])]\n        rendered=tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)\n        ids=tokenizer.apply_chat_template(messages,tokenize=True,add_generation_prompt=True,return_tensors='pt').to('cuda')\n        kwargs=dict(do_sample=False,max_new_tokens=MAX_NEW,pad_token_id=tokenizer.eos_token_id)\n        if force: kwargs['min_new_tokens']=MAX_NEW\n        clock=TokenClock(); torch.cuda.synchronize(); begin=time.perf_counter()\n        with torch.inference_mode(): result=model.generate(ids,streamer=clock,**kwargs)\n        torch.cuda.synchronize(); end=time.perf_counter()\n        selected=result[0,ids.shape[-1]:].tolist()\n        input_ids=ids[0].tolist()\n        if clock.first is None or not selected: raise RuntimeError('No generated token callback')\n        row=dict(prompt_tokens=len(input_ids),input_token_ids=input_ids,rendered_chat=rendered,\n                 output_tokens=len(selected),output_token_ids=selected,\n                 output=tokenizer.decode(selected,skip_special_tokens=True),\n                 output_token_pieces=tokenizer.convert_ids_to_tokens(selected),\n                 stop_reason='fixed_length' if force else ('eos' if selected[-1] in eos_ids else 'length'),\n                 ttft_s=clock.first-begin,wall_s=end-begin,\n                 post_first_s_per_token=(end-clock.first)/(len(selected)-1) if len(selected)>1 else None)\n        del result,ids\n        return row\n    data=dict(contract=PROTOCOL,status='running',utc_started=datetime.now(timezone.utc).isoformat(),\n              model_id=MODEL_ID,revision=REVISION,gpu=torch.cuda.get_device_name(0),\n              torch=torch.__version__,cuda=torch.version.cuda,transformers=transformers.__version__,\n              bitsandbytes=bitsandbytes.__version__,trials=TRIALS,max_new_tokens=MAX_NEW,\n              quality_mode='natural_eos',timing_mode='fixed_output_length',formats={},resource_samples=[preflight],\n              clock='synchronized generate wall; first generated-ID streamer callback; post-first proxy includes return overhead, not last-emission interval; not HTTP or pure GPU time',\n              quantization=dict(quant_type='nf4',compute_dtype='float16',double_quant=False),\n              limitations=['four-prompt smoke, not quality benchmark','fixed format order, three trials, uncontrolled timing variance',\n                           'allocated/reserved are PyTorch allocator counters, not complete process/device memory'])\n    checkpoint(output_path,data)\n    model=None\n    try:\n        for label in ('fp16','nf4_w4a16'):\n            resource_gate(); gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()\n            before=torch.cuda.memory_allocated()\n            quant=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_compute_dtype=torch.float16,bnb_4bit_use_double_quant=False) if label=='nf4_w4a16' else None\n            begin=time.perf_counter()\n            model=AutoModelForCausalLM.from_pretrained(str(snapshot),local_files_only=True,torch_dtype=torch.float16,\n                    quantization_config=quant,device_map={'':0}).eval()\n            torch.cuda.synchronize()\n            eos=model.generation_config.eos_token_id\n            eos_ids=eos if isinstance(eos,list) else [eos]\n            entry=dict(load_s=time.perf_counter()-begin,rows=[],status='running',baseline_allocated_bytes=before,\n                       eos_token_ids=eos_ids,quantized_linear_modules=sum(type(m).__name__=='Linear4bit' for m in model.modules()))\n            data['formats'][label]=entry; checkpoint(output_path,data)\n            _=one(model,PROMPTS[0],True)  # Exactly one warmup per format, excluded.\n            for case in PROMPTS:\n                row=dict(bucket=case['bucket'],prompt=case['text'],quality_sample=one(model,case,False),timing_trials=[])\n                entry['rows'].append(row); checkpoint(output_path,data)\n                for trial in range(TRIALS):\n                    resource_gate(); row['timing_trials'].append(one(model,case,True)); checkpoint(output_path,data)\n                for field in ('ttft_s','wall_s'):\n                    row['p95_'+field]=max(t[field] for t in row['timing_trials'])\n                    row['median_'+field]=statistics.median(t[field] for t in row['timing_trials'])\n                checkpoint(output_path,data)\n            entry.update(peak_allocated_bytes=torch.cuda.max_memory_allocated(),peak_reserved_bytes=torch.cuda.max_memory_reserved(),status='complete')\n            del model; model=None; gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()\n            entry['post_cleanup_allocated_bytes']=torch.cuda.memory_allocated()\n            if entry['post_cleanup_allocated_bytes']>before+64*2**20: raise RuntimeError('STOP: live allocation did not recover')\n            entry['cleanup_verified']=True\n            data['resource_samples'].append(resource_gate()); checkpoint(output_path,data)\n        data['status']='complete'\n    except BaseException as exc:\n        data['status']='failed'; data['error_type']=type(exc).__name__; raise\n    finally:\n        if model is not None: del model\n        gc.collect(); torch.cuda.empty_cache()\n        data['utc_finished']=datetime.now(timezone.utc).isoformat(); checkpoint(output_path,data)\n    return data\n\n\ndef validate_artifact(data):\n    def require(value,message):\n        if not value: raise ValueError(message)\n    require(data.get('contract')==PROTOCOL and data.get('status')=='complete','incomplete or wrong protocol')\n    require(data.get('revision')==REVISION and data.get('model_id')==MODEL_ID,'model identity')\n    require(data.get('transformers')=='4.51.3' and data.get('bitsandbytes')=='0.50.2','runtime pins')\n    require(all(isinstance(data.get(key),str) and data[key] for key in ('torch','cuda','gpu')),'runtime identity')\n    require(data.get('quantization')==dict(quant_type='nf4',compute_dtype='float16',double_quant=False),'quantization config')\n    require(data.get('trials')==TRIALS and data.get('max_new_tokens')==MAX_NEW,'trial budget')\n    require(data.get('quality_mode')=='natural_eos' and data.get('timing_mode')=='fixed_output_length','separate quality/timing')\n    require(set(data.get('formats',{}))=={'fp16','nf4_w4a16'},'paired formats')\n    paired=[]\n    for label in ('fp16','nf4_w4a16'):\n        entry=data['formats'][label]\n        require(entry.get('status')=='complete' and len(entry.get('rows',[]))==4,'complete format rows')\n        require(entry.get('quantized_linear_modules',-1)==0 if label=='fp16' else entry.get('quantized_linear_modules',0)>0,'actual module treatment')\n        require(all(type(entry.get(key)) is int and entry[key]>=0 for key in ('peak_allocated_bytes','peak_reserved_bytes','baseline_allocated_bytes','post_cleanup_allocated_bytes')),'finite integer byte counters')\n        require(0<entry['peak_allocated_bytes']<=entry['peak_reserved_bytes'],'allocator counters')\n        require(entry.get('cleanup_verified') is True and entry['post_cleanup_allocated_bytes']<=entry['baseline_allocated_bytes']+64*2**20,'cleanup verified')\n        require(math.isfinite(entry['load_s']) and entry['load_s']>0,'load clock')\n        inputs=[]\n        for case,row in zip(PROMPTS,entry['rows']):\n            require(row['bucket']==case['bucket'] and row['prompt']==case['text'],'paired literal prompt')\n            require(len(row['timing_trials'])==TRIALS,'three raw timing trials')\n            samples=[row['quality_sample']]+row['timing_trials']\n            for index,sample in enumerate(samples):\n                require(sample['prompt_tokens']==len(sample['input_token_ids'])>0,'prompt IDs/count')\n                require(sample['output_tokens']==len(sample['output_token_ids'])==len(sample['output_token_pieces'])>0,'output IDs/count')\n                require(all(type(x) is int and x>=0 for x in sample['input_token_ids']+sample['output_token_ids']),'token ID types')\n                require(sample['input_token_ids']==samples[0]['input_token_ids'] and sample['rendered_chat']==samples[0]['rendered_chat'],'within-prompt identity')\n                require(math.isfinite(sample['wall_s']) and math.isfinite(sample['ttft_s']) and 0<sample['ttft_s']<=sample['wall_s'],'finite ordered clocks')\n                expected=(sample['wall_s']-sample['ttft_s'])/(sample['output_tokens']-1) if sample['output_tokens']>1 else None\n                require(sample['post_first_s_per_token']==expected,'post-first denominator')\n                require(sample['output_tokens']<=MAX_NEW,'bounded output')\n                if index: require(sample['output_tokens']==MAX_NEW and sample['stop_reason']=='fixed_length','fixed timing length')\n                else:\n                    is_eos=sample['output_token_ids'][-1] in entry['eos_token_ids']\n                    require(sample['stop_reason']==('eos' if is_eos else 'length'),'natural stop reason')\n                    require(is_eos or sample['output_tokens']==MAX_NEW,'natural termination requires EOS or cap')\n            for field in ('ttft_s','wall_s'):\n                values=[sample[field] for sample in row['timing_trials']]\n                require(row['p95_'+field]==max(values) and row['median_'+field]==statistics.median(values),'raw summary arithmetic')\n            inputs.append((samples[0]['input_token_ids'],samples[0]['rendered_chat']))\n        paired.append(inputs)\n    require(paired[0]==paired[1],'identical tokenized inputs across formats')\n    return True\n",lab_namespace)
print('Read-only headroom:',lab_namespace['resource_gate'](require_idle=True))
MODEL_DIR=lab_namespace['resolve_snapshot']()
print('Immutable local snapshot resolved; keep local paths private.')


## 3. One bounded run, checkpoint after every sample/trial

Each format loads separately, warms once outside timing, then runs the four cases. FP16 always precedes NF4: fixed order and three trials are limitations, not a controlled production speed claim. The measured wall interval covers generate plus Python/host overhead; each streamer callback synchronizes CUDA. Model load and chat tokenization are outside the timed request.

Resource failure, OOM, timeout, missing callback or unrecovered live allocation stops immediately. Partial evidence is saved with failed/running status and rejected as complete. Preserve it, inspect the last stage, and diagnose before retrying. Do not restart all notebooks or delete caches. The model is released in finally.

The evidence path is fresh per attempt. After success copy only the validated JSON into evidence/topic-03-comparison.json for submission. On instructor hardware use the external bounded runner, not an unbounded notebook execution.


In [ ]:
from pathlib import Path
from datetime import datetime,timezone
ATTEMPT=Path('evidence')/datetime.now(timezone.utc).strftime('topic03-%Y%m%dT%H%M%S%fZ')
ATTEMPT.mkdir(parents=True,exist_ok=False)
RUN=lab_namespace['run_lab'](MODEL_DIR,ATTEMPT/'comparison.json')
assert lab_namespace['validate_artifact'](RUN)
print('Validated complete evidence:',ATTEMPT/'comparison.json')


## 4. Validate, annotate and decide

The validator checks identity, paired inputs, exact counts, finite clocks, EOS/fixed-length separation, raw-to-summary arithmetic, actual quantized modules and allocator counters. Passing it does **not** prove semantic quality or accelerated kernels.

Submit evidence/topic-03-comparison.json and evidence/topic-03-decision.md. In the memo: (1) derive weight payload + scale metadata and missing runtime terms; (2) annotate all eight natural samples; (3) compare raw clocks and peak allocated/reserved memory without calling either full process VRAM; (4) explain one failed prediction; (5) make a conditional go/no-go decision and name a discriminating held-out/concurrency experiment. Three-sample nearest-rank p95 equals the observed maximum, not a reliable tail estimate. No HTTP TTFT/TPOT/goodput was measured.

Keep the original attempt and source revision. Export no model caches, local server logs, tokens or private roster data. Standalone self-check: python scripts/validate_topic03.py evidence/topic-03-comparison.json in the public template.
